# King County Housing Price Prediction with Economic Indicators — XGBoost vs TabNet

This notebook reproduces the end-to-end analysis used in the report and extends it with external economic indicators from FRED. It trains two model families (XGBoost and TabNet) under two feature settings for **King County as one housing market**:

- **PHYSICAL** (structure-only): size, grade, beds/baths, age, etc.
- **FULL** (structure + location + economic context): PHYSICAL plus distances, views, submarkets, time effects, mortgage rates, local CPI, unemployment, and King County income.

**Scope:** King County, Washington as a whole; Seattle is retained as a city/submarket indicator rather than treated as a separate project.  
**Validation:** 5-fold cross-validation; target is log1p(price).

## Quick Start (notebook)
1. Run the setup/load cells and point `DATA_FILES` to the Kaggle/Google Drive housing CSVs.
2. Run the economic-indicator merge cell to add FRED mortgage, CPI, unemployment, and income variables.
3. Execute cells section-by-section.

## Notebook Outline
1. Setup & Config
2. Load & Clean Housing Sales
3. Add External Economic Indicators
4. Feature Engineering (minimal, leak-free)
5. XGBoost — FULL & PHYSICAL
6. TabNet — FULL & PHYSICAL
7. Cross-validated Results (log and $)
8. Best-model Figures (OOF scatter; feature-importance)
9. Save tables/figures


In [ ]:
# ==========================
# LOCAL LOAD: KING COUNTY HOUSING + FINANCIAL INDICATORS
# ==========================
# This local Jupyter version uses the merged CSV already built from:
# - Andy Krause King County sales data
# - FRED mortgage, CPI, unemployment, and income indicators

import pandas as pd

CLEAN_PATH = r"/path/to/local/Documents/Codex/2026-10-06/us/outputs/kingco_sales_last10yrs_with_financial_indicators.csv"
ECON_PATH = CLEAN_PATH
MERGED_PATH = CLEAN_PATH

df_final = pd.read_csv(CLEAN_PATH, low_memory=False)
df_final["sale_date"] = pd.to_datetime(df_final["sale_date"], errors="coerce")

print(f"Loaded local enriched dataset: {CLEAN_PATH}")
print(f"Date range: {df_final['sale_date'].min().date()} → {df_final['sale_date'].max().date()}")
print(f"Shape: {df_final.shape}")
print("Economic columns present:", [
    c for c in [
        "mortgage_30yr_rate",
        "cpi_seattle_all_items",
        "unemployment_rate_seattle_msa",
        "king_county_per_capita_income",
    ] if c in df_final.columns
])



In [ ]:
# --------------------------
# Reload cleaned dataset (df_final only)
# --------------------------
df_final = pd.read_csv(CLEAN_PATH, low_memory=False)
df_final['sale_date'] = pd.to_datetime(df_final['sale_date'], errors='coerce')

# Display basic dataset checks
print("\n=== Columns ===")
print(df_final.columns.tolist())

print("\n=== First 3 Rows ===")
print(df_final.head(3))

# Confirm date range and dataset shape
print(f"\n📅 Date window: {df_final['sale_date'].min().date()} → {df_final['sale_date'].max().date()}")
print(f"🧮 Shape: {df_final.shape}")


In [ ]:
# ==========================
# ADD EXTERNAL ECONOMIC / FINANCIAL INDICATORS FROM FRED
# ==========================
indicator_cols = [
    "mortgage_30yr_rate",
    "cpi_seattle_all_items",
    "unemployment_rate_seattle_msa",
    "king_county_per_capita_income",
]

if all(col in df_final.columns for col in indicator_cols):
    print("External economic indicators already present; skipping FRED re-merge.")
    print((df_final[indicator_cols].isna().mean() * 100).round(2).astype(str) + "%")
else:
    import pandas as pd

    FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"

    FRED_SERIES = {
        "MORTGAGE30US": "mortgage_30yr_rate",
        "CUUSA423SA0": "cpi_seattle_all_items",
        "SEAT653URN": "unemployment_rate_seattle_msa",
        "PCPI53033": "king_county_per_capita_income",
    }

    def read_fred(series_id, output_name):
        fred = pd.read_csv(FRED_CSV.format(series_id=series_id))
        fred["DATE"] = pd.to_datetime(fred["observation_date"], errors="coerce")
        fred[series_id] = pd.to_numeric(fred[series_id], errors="coerce")
        fred = fred.dropna(subset=["DATE", series_id]).copy()
        return fred.rename(columns={series_id: output_name})

    def monthly_average(series_id, output_name):
        fred = read_fred(series_id, output_name)
        fred["sale_month_start"] = fred["DATE"].dt.to_period("M").dt.to_timestamp()
        return fred.groupby("sale_month_start", as_index=False)[output_name].mean()

    def monthly_last(series_id, output_name):
        fred = read_fred(series_id, output_name)
        fred["sale_month_start"] = fred["DATE"].dt.to_period("M").dt.to_timestamp()
        return fred[["sale_month_start", output_name]].drop_duplicates("sale_month_start", keep="last")

    def asof_by_month(df, external):
        return pd.merge_asof(
            df.sort_values("sale_month_start"),
            external.sort_values("sale_month_start"),
            on="sale_month_start",
            direction="backward",
        ).sort_index()

    def asof_by_date(df, external, value_col):
        return pd.merge_asof(
            df.sort_values("sale_date"),
            external[["DATE", value_col]].sort_values("DATE"),
            left_on="sale_date",
            right_on="DATE",
            direction="backward",
        ).drop(columns=["DATE"]).sort_index()

    df_final["sale_date"] = pd.to_datetime(df_final["sale_date"], errors="coerce")
    df_final = df_final.dropna(subset=["sale_date"]).copy()
    df_final["sale_year"] = df_final["sale_date"].dt.year
    df_final["sale_month"] = df_final["sale_date"].dt.month
    df_final["sale_month_start"] = df_final["sale_date"].dt.to_period("M").dt.to_timestamp()

    mortgage_monthly = monthly_average("MORTGAGE30US", FRED_SERIES["MORTGAGE30US"])
    unemployment_monthly = monthly_last("SEAT653URN", FRED_SERIES["SEAT653URN"])
    cpi = read_fred("CUUSA423SA0", FRED_SERIES["CUUSA423SA0"])
    income = read_fred("PCPI53033", FRED_SERIES["PCPI53033"])

    df_final = asof_by_month(df_final, mortgage_monthly)
    df_final = asof_by_month(df_final, unemployment_monthly)
    df_final = asof_by_date(df_final, cpi, FRED_SERIES["CUUSA423SA0"])
    df_final = asof_by_date(df_final, income, FRED_SERIES["PCPI53033"])
    df_final = df_final.drop(columns=["sale_month_start"])

    indicator_cols = list(FRED_SERIES.values())
    print("External economic indicators added:")
    print(indicator_cols)
    print("Missing percentage after merge:")
    print((df_final[indicator_cols].isna().mean() * 100).round(2).astype(str) + "%")
    print(f"Updated shape: {df_final.shape}")

    ECON_PATH = "/content/drive/MyDrive/king county sales/kc_housing_last10yrs_clean_with_economic_indicators.csv"
    df_final.to_csv(ECON_PATH, index=False)
    print(f"Economic-indicator dataset saved: {ECON_PATH}")



In [ ]:
# --------------------------
# Rename columns to human-friendly names
# --------------------------

# Mapping of raw column names → descriptive names
rename_map = {
    'sale_id': 'transaction_id',
    'pinx': 'property_id',
    'sale_date': 'sale_date',
    'sale_price': 'sale_price_usd',
    'sale_nbr': 'sale_count_for_property',
    'sale_warning': 'sale_warning_code',
    'join_status': 'join_type',
    'join_year': 'data_join_year',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'area': 'assessment_area',
    'city': 'city',
    'zoning': 'zoning_code',
    'subdivision': 'subdivision_name',
    'present_use': 'property_use_type',
    'land_val': 'land_value',
    'imp_val': 'building_value',
    'year_built': 'year_built',
    'year_reno': 'year_renovated',
    'sqft_lot': 'lot_sqft',
    'sqft': 'total_living_sqft',
    'sqft_1': 'main_floor_sqft',
    'sqft_fbsmt': 'finished_basement_sqft',
    'grade': 'building_grade',
    'fbsmt_grade': 'basement_grade',
    'condition': 'home_condition',
    'stories': 'number_of_stories',
    'beds': 'bedrooms',
    'bath_full': 'full_bathrooms',
    'bath_3qtr': 'three_quarter_bathrooms',
    'bath_half': 'half_bathrooms',
    'garb_sqft': 'basement_garage_sqft',
    'gara_sqft': 'attached_garage_sqft',
    'wfnt': 'waterfront_rating',
    'golf': 'on_golf_course',
    'greenbelt': 'on_greenbelt',
    'noise_traffic': 'traffic_noise_rating',
    'view_rainier': 'view_mountain_rainier',
    'view_olympics': 'view_olympic_mountains',
    'view_cascades': 'view_cascade_mountains',
    'view_territorial': 'view_territorial',
    'view_skyline': 'view_seattle_skyline',
    'view_sound': 'view_puget_sound',
    'view_lakewash': 'view_lake_washington',
    'view_lakesamm': 'view_lake_sammamish',
    'view_otherwater': 'view_other_water',
    'view_other': 'view_other',
    'submarket': 'submarket_area'
}

# Apply renaming
df_final = df_final.rename(columns=rename_map)
print("✅ Columns renamed to human-friendly format.")

# Preview first 3 rows
print(df_final.head(3))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# 1. Set style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 12

# 2. Filter data up to 2025 and prepare
df_plot = (df_final
           # Ensure year is extracted
           .assign(sale_year=lambda x: x['sale_date'].dt.year)
           # Explicitly filter to 2013–2025
           .query('2013 <= sale_year <= 2025')
           # Create clear region labels
           .assign(region=lambda x: x['is_seattle'].map({0: 'King County', 1: 'Seattle'}))
           # Calculate metrics
           .groupby(['sale_year', 'region'], as_index=False)
           .agg(median_price=('sale_price_usd', 'median')))  # ✅ use renamed column

# 3. Create plot
plt.figure(figsize=(12,6))
ax = sns.lineplot(
    data=df_plot,
    x='sale_year',
    y='median_price',
    hue='region',
    style='region',
    markers=['o', 's'],
    dashes=False,
    palette=['#1f77b4', '#d62728'],  # Blue/red
    linewidth=2,
    markersize=8
)

# 4. Formatting
def currency_format(x, pos):
    """Custom formatter: $1.1M instead of $1,100,000"""
    if x >= 1e6:
        return f'${x/1e6:.1f}M'
    return f'${x/1e3:.0f}K'

ax.set(
    title='King County Home Prices (2013–2025)',
    xlabel='Year',
    ylabel='Median Price',
    xlim=(2012.5, 2025.5)  # Pad years slightly
)
ax.yaxis.set_major_formatter(FuncFormatter(currency_format))

# 5. Customize legend and ticks
ax.legend(title='Region', frameon=True, shadow=True)
ax.set_xticks(range(2013, 2026, 2))  # Show even years

plt.tight_layout()
plt.savefig('price_trends_2013-2025.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Select numeric columns (exclude IDs and binary flags)
exclude_cols = ['transaction_id', 'property_id', 'is_seattle']
numeric_cols = [
    col for col in df_final.select_dtypes(include=['float64', 'int64']).columns
    if col not in exclude_cols
]

# Plot histograms for numeric columns
plt.figure(figsize=(15, 16))
df_final[numeric_cols].hist(
    figsize=(15, 16),
    bins=30,
    edgecolor='black',
    grid=False
)

# Add a clear main title and adjust layout
plt.suptitle("Histograms of Numeric Features", fontsize=18, y=1.02, weight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])  # keep title visible
plt.show()


In [ ]:
# --------------------------
# Missing data analysis (scans ALL columns)
# --------------------------
import matplotlib.pyplot as plt

def analyze_missing_data(df, print_summary=True, plot_distribution=True, top_n=40):
    """
    Analyze missing values across all columns and (optionally) plot the distribution.

    Args:
        df: DataFrame to analyze (all columns are scanned)
        print_summary: Print a concise summary table
        plot_distribution: Show a bar chart of missing percentages
        top_n: Limit the plot/table to the top N columns with most missing values (for readability)

    Returns:
        DataFrame with columns: missing_count, missing_pct, dtype (only columns with missing > 0)
    """
    # Compute missing stats for all columns
    stats_all = (
        df.isna().sum()
        .to_frame('missing_count')
        .assign(
            missing_pct=lambda x: (x['missing_count'] / len(df) * 100),
            dtype=df.dtypes
        )
        .sort_values('missing_count', ascending=False)
    )

    # Keep only columns that have missing values (> 0)
    missing_stats = stats_all.query('missing_count > 0')

    if print_summary:
        total_cols = df.shape[1]
        cols_with_missing = len(missing_stats)
        cols_without_missing = total_cols - cols_with_missing
        total_missing_cells = int(stats_all['missing_count'].sum())

        print("=== MISSING VALUE ANALYSIS ===")
        print(f"Columns with missing values: {cols_with_missing} / {total_cols}")
        print(f"Columns with NO missing values: {cols_without_missing}")
        print(f"Total missing cells: {total_missing_cells:,}")

        if cols_with_missing > 0:
            print("\nTop columns by missing values:")
            display_df = (
                missing_stats.head(top_n)
                .assign(
                    missing_pct=lambda x: x['missing_pct'].round(1),
                    missing_count=lambda x: x['missing_count'].map('{:,}'.format)
                )
                [['missing_count', 'missing_pct', 'dtype']]
            )
            print(display_df.to_string())

    if plot_distribution and not missing_stats.empty:
        # Plot top N columns by missing percentage
        plot_df = missing_stats.head(top_n).sort_values('missing_pct', ascending=True)
        ax = plot_df['missing_pct'].plot(kind='barh', figsize=(10, 6))
        ax.set_title(f'Missing Values by Column (Top {min(top_n, len(missing_stats))}, {len(df):,} rows)')
        ax.set_xlabel('Percentage Missing')
        ax.set_ylabel('')
        # Improve label readability
        plt.tight_layout()
        plt.show()

    return missing_stats

# Execute analysis
missing_stats = analyze_missing_data(df_final, print_summary=True, plot_distribution=True, top_n=40)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from textwrap import shorten

# Global plot settings (readable on screen and in docs)
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 120

def plot_categorical_overview(
    df: pd.DataFrame,
    max_uniques: int = 1000,   # include columns with <= this many unique values
    top_k: int = 15,           # show top K categories; bucket the rest as "Other"
    max_plots: int = 12,       # max number of columns to plot
    n_cols: int = 2,           # subplot grid columns
    title: str = "Categorical Feature Distributions",
    label_top: int = 5         # add value labels for top N bars
):
    """
    Overview of categorical-like columns (object/category types).
    Scans all such columns with 2..max_uniques categories.
    """
    # Candidate categorical columns (scans ALL object/category cols)
    cat_cols = [
        c for c in df.select_dtypes(include=['object', 'category']).columns
        if 2 <= df[c].nunique(dropna=False) <= max_uniques
    ]
    if not cat_cols:
        print("No categorical columns found under the current filters.")
        return

    # Highest-cardinality first; cap number of plots
    cat_cols = sorted(cat_cols, key=lambda c: df[c].nunique(dropna=False), reverse=True)[:max_plots]

    n_rows = math.ceil(len(cat_cols) / n_cols)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(15, 4*n_rows),
        squeeze=False,
        constrained_layout=True  # prevents overlap without manual tweaking
    )
    fig.suptitle(title, y=1.02, fontsize=16, weight='bold')

    for ax, col in zip(axes.flatten(), cat_cols):
        # Counts including NA
        counts = (
            df[col].astype('object').fillna('<NA>')
              .value_counts(dropna=False)
              .sort_values(ascending=False)
        )

        # Keep top_k; bucket rest as "Other"
        if len(counts) > top_k:
            top = counts.iloc[:top_k]
            other_sum = counts.iloc[top_k:].sum()
            counts = pd.concat([top, pd.Series({'Other': other_sum})])

        plot_df = counts.reset_index()
        plot_df.columns = [col, 'count']
        plot_df[col] = plot_df[col].astype(str).apply(lambda s: shorten(s, width=35, placeholder='…'))

        # Single-color bars for clarity
        sns.barplot(
            data=plot_df, y=col, x='count', ax=ax,
            color='tab:blue', edgecolor='black', linewidth=0.5
        )

        uniq = int(df[col].nunique(dropna=False))
        ax.set_title(f"{col}  (uniq={uniq}, top {min(top_k, uniq)})", pad=8)
        ax.set_xlabel('Count')
        ax.set_ylabel('')
        ax.grid(axis='x', linestyle='--', alpha=0.3)

        # Label the top bars
        for i, p in enumerate(ax.patches):
            if i < min(label_top, len(ax.patches)):
                x = p.get_width()
                y = p.get_y() + p.get_height()/2
                ax.annotate(
                    f"{int(x):,}", (x, y),
                    xytext=(3, 0), textcoords='offset points',
                    va='center', ha='left', fontsize=9
                )

    # Hide unused axes (if any)
    for ax in axes.flatten()[len(cat_cols):]:
        ax.set_visible(False)

    plt.show()

# ---- run it ----
plot_categorical_overview(
    df_final,
    max_uniques=1000,
    top_k=15,
    max_plots=12,
    n_cols=2,
    label_top=5
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import shorten

# Style (clean and readable)
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 12

# Column to clean + analyze
col = 'submarket_area'
if col not in df_final.columns:
    raise KeyError(f"Column '{col}' not found in df_final")

# CLEAN IN df_final (drop rows where the target column is NaN)
rows_before = len(df_final)
nan_before = int(df_final[col].isna().sum())
df_final = df_final[df_final[col].notna()].copy()
rows_after = len(df_final)
print(f"🧹 Cleaned '{col}' in df_final: rows {rows_before:,} → {rows_after:,} (removed {nan_before:,} NaNs)")

# Limit to top categories for readability
top_n = 30
order = df_final[col].value_counts().head(top_n).index

# shorten very long labels for a cleaner figure (no helper column added to df_final)
label_map = {v: shorten(str(v), width=40, placeholder='…') for v in order}
labels = df_final[col].map(label_map)

# Build the plot
plt.figure(figsize=(12, 8), constrained_layout=True)
ax = sns.countplot(
    y=labels,
    order=[label_map[v] for v in order],
    edgecolor='black',
    linewidth=0.5
)

ax.set_title(f"Submarket Distribution (Top {min(top_n, len(order))}; NaNs Removed in df_final)", pad=8)
ax.set_xlabel("Property Count")
ax.set_ylabel("")
ax.grid(axis='x', linestyle='--', alpha=0.3)

# Add value and percent labels for each bar
counts = labels.value_counts()
total = int(counts.sum())

for p in ax.patches:
    count = int(p.get_width())
    pct = (count / total) * 100 if total else 0
    y = p.get_y() + p.get_height() / 2
    ax.annotate(f"{count:,}  ({pct:.1f}%)",
                xy=(count, y),
                xytext=(5, 0),
                textcoords='offset points',
                va='center', ha='left', fontsize=10)

# Slight padding so labels don’t clip
xmax = max([p.get_width() for p in ax.patches]) if ax.patches else 0
ax.set_xlim(0, xmax * 1.08 if xmax else 1)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 12

def analyze_and_clean(df, column='subdivision_name', top_n=20):
    """
    Missing-data check + simple cleaning + clear bar chart for the top categories.

    Args:
        df: Input DataFrame
        column: Column to analyze/clean
        top_n: Number of top categories to display after cleaning

    Returns:
        cleaned_df: DataFrame with rows containing NaN in `column` removed
    """
    # Validate column
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame")

    # 1) Pre-cleaning summary
    missing_before = df[column].isna().sum()
    pct_missing = (missing_before / len(df) * 100) if len(df) else 0

    print("=== MISSING DATA ANALYSIS ===")
    print(f"• Column: {column}")
    print(f"• Missing values: {missing_before:,} ({pct_missing:.1f}%)")
    print(f"• Total rows before: {len(df):,}")

    # 2) Recommendation (informational only)
    if pct_missing > 5:
        print("\n⚠️  Recommendation: Consider keeping missing as its own category.")
    else:
        print("\n✅ Recommendation: Safe to drop missing values (low %).")

    # 3) Cleaning (drop rows with missing in the target column)
    cleaned_df = df.dropna(subset=[column]).copy()
    rows_removed = len(df) - len(cleaned_df)
    remain_pct = (len(cleaned_df) / len(df) * 100) if len(df) else 0

    print("\n🧹 Cleaning results:")
    print(f"• Removed rows: {rows_removed:,}")
    print(f"• Remaining rows: {len(cleaned_df):,} ({remain_pct:.1f}% of original)")

    # 4) Visualization (post-cleaning)
    counts = cleaned_df[column].value_counts().head(top_n)
    total = int(counts.sum())

    # Plot (horizontal bar for readability)
    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    sns.barplot(
        x=counts.values,
        y=counts.index,
        edgecolor='black',
        linewidth=0.5,
        ax=ax
    )

    ax.set_title(f"Top {top_n} {column.replace('_', ' ').title()} (After Cleaning)", pad=8)
    ax.set_xlabel("Count")
    ax.set_ylabel("")
    ax.grid(axis='x', linestyle='--', alpha=0.3)

    # Labels: count + percent
    xmax = counts.max() if not counts.empty else 0
    for i, v in enumerate(counts.values):
        pct = (v / total * 100) if total else 0
        ax.annotate(
            f"{v:,}  ({pct:.1f}%)",
            xy=(v, i),
            xytext=(6, 0),
            textcoords='offset points',
            va='center', ha='left', fontsize=10
        )

    # Add slight right padding so labels don’t clip
    ax.set_xlim(0, xmax * 1.08 if xmax else 1)

    # Footnote
    fig.text(
        0.5, -0.02,
        f"Note: Removed {rows_removed:,} rows ({pct_missing:.1f}%) with missing '{column}'.",
        ha='center', va='top', fontsize=10, color='gray'
    )

    plt.show()
    return cleaned_df

# Run
df_final = analyze_and_clean(df_final, column='subdivision_name', top_n=20)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def cleanJoinType(df):
    """
    Standardize and clean 'join_type' with clear reporting:
      - Raw (pre-mapping)
      - Mapped (pre-drop)
      - Final kept (post-drop)
    """
    col = 'join_type'
    if col not in df.columns:
        print(f"⚠️ Column '{col}' not found. Available columns: {df.columns.tolist()}")
        return df

    # Keep a copy of the raw values for reporting
    raw = df[col].copy()

    # Normalize and map to canonical categories
    ser = (df[col].astype(str).str.strip().str.lower().str.replace(r'\s+', ' ', regex=True))
    toCanonical = {
        'nochg':'unchanged','no change':'unchanged',
        'new':'new',
        'reno - before':'modified','reno - after':'modified',
        'rebuilt - before':'modified','rebuilt - after':'modified',
        'demo':'demolished','demolition':'demolished','demolished':'demolished',
        'miss99':'unknown','missing':'unknown','na':'unknown','none':'unknown'
    }
    mapped = ser.map(toCanonical).fillna('unknown')

    # Report raw distribution (pre-mapping)
    print("\n=== Join Type (raw, before mapping) ===")
    print(raw.value_counts(dropna=False))

    # Apply mapping to the DataFrame
    df = df.copy()
    df[col] = mapped

    # Report mapped distribution (pre-drop)
    print("\n=== Join Type (mapped, before dropping) ===")
    print(df[col].value_counts(dropna=False))

    # Drop rows
    before = len(df)
    df = df.loc[~df[col].isin(['demolished','unknown'])].copy()
    dropped = before - len(df)
    print(f"\n🧹 Dropped {dropped:,} rows ('demolished' + 'unknown')")

    # Report final kept distribution (post-drop)
    print("\n=== Join Type (final kept categories) ===")
    print(df[col].value_counts(dropna=False))

    # Plot final distribution
    plt.figure(figsize=(8,5))
    sns.countplot(
        y=col,
        data=df,
        order=df[col].value_counts().index,
        palette="Blues_r",
        edgecolor="black",
        linewidth=0.5
    )
    plt.title("Join Type Distribution (Cleaned)", pad=10, fontsize=14, weight='bold')
    plt.xlabel("Count"); plt.ylabel("")
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

    return df

# Run
df_final = cleanJoinType(df_final)


In [ ]:
# === Remove price-leakers + admin/ID/meta columns (in-place on df_final) ===
import re

assert 'df_final' in globals(), "df_final not found."

# 1) Price-leaking columns (must drop, they reveal target info)
PRICE_LEAKERS = [
    'sale_count_for_property',   # future info
    'land_val', 'imp_val',       # pre-rename
    'land_value', 'building_value',  # post-rename
    # examples of others to guard against:
    # 'price_per_sqft', 'last_sale_price', 'assessed_value'
]

# 2) Admin / ID / meta columns (not useful for modeling)
ADMIN_META = [
    'transaction_id', 'sale_id', 'property_id', 'pinx',   # IDs / keys
    'data_join_year', 'property_use_desc',                # provenance/admin
    'Unnamed: 0', 'Unnamed: 1'                            # auto-index junk
]

# 3) Pattern-based cleanup
ADMIN_PREFIXES = ['_data_source', '_file_source', 'source_']  # drop any col starting with these
EXCLUDE_SAFE = set(['grid'])  # keep exceptions even if pattern matches

def matches_patterns(col):
    if any(col.startswith(pfx) for pfx in ADMIN_PREFIXES):
        return True
    if col.endswith('_id') and col not in EXCLUDE_SAFE:
        return True
    return False

pattern_hits = [c for c in df_final.columns if matches_patterns(c)]

# 4) Build drop list
drop_list = []
drop_list += [c for c in PRICE_LEAKERS if c in df_final.columns]
drop_list += [c for c in ADMIN_META if c in df_final.columns]
drop_list += pattern_hits

# 5) Drop 'sale_date' only if year/month already exist
if 'sale_date' in df_final.columns:
    if {'sale_year', 'sale_month'}.issubset(df_final.columns):
        drop_list.append('sale_date')
    else:
        print("⏭️ Kept 'sale_date' (year/month not available).")

# 6) Execute drop
drop_list = sorted(set(drop_list))
cols_present = [c for c in drop_list if c in df_final.columns]

c_before, r_before = df_final.shape[1], df_final.shape[0]
df_final.drop(columns=cols_present, inplace=True, errors='ignore')
c_after, r_after = df_final.shape[1], df_final.shape[0]

# 7) Report
print("✅ Removed leakers + admin/ID/meta from df_final.")
print(f"   Dropped ({len(cols_present)}): {cols_present if cols_present else 'None'}")
print(f"   Columns: {c_before} → {c_after}  |  Rows unchanged: {r_after:,}")

# Final sanity check
remaining_leakers = [c for c in PRICE_LEAKERS if c in df_final.columns]
print("✔️ Confirmed: No price-leakers remain."
      if not remaining_leakers else f"❌ Still present: {remaining_leakers}")


In [ ]:
# 🔍 Strange numeric values (negative, zero where impossible)
numeric_cols = df_final.select_dtypes(include=['int64', 'float64']).columns
strange_values = {}

for col in numeric_cols:
    negatives = (df_final[col] < 0).sum()
    zeros = (df_final[col] == 0).sum()
    if negatives > 0 or zeros > 0:
        strange_values[col] = {'negatives': negatives, 'zeros': zeros}

print("=== Strange Numeric Values ===")
for col, vals in strange_values.items():
    print(f"{col}: {vals['negatives']} negatives, {vals['zeros']} zeros")


In [ ]:
# 🔍 Dates outside range
if 'sale_date' in df_final.columns:
    min_date = df_final['sale_date'].min()
    max_date = df_final['sale_date'].max()
    print(f"Sale Date Range: {min_date} to {max_date}")

if 'year_built' in df_final.columns:
    future_years = df_final[df_final['year_built'] > 2025]
    print(f"Properties with future build year: {len(future_years)}")

if 'year_renovated' in df_final.columns:
    future_reno = df_final[df_final['year_renovated'] > 2025]
    print(f"Properties with future renovation year: {len(future_reno)}")


In [ ]:
import numpy as np

# Exclude ID/flag columns that aren’t useful for variance/skew checks
exclude = {'transaction_id', 'property_id', 'is_seattle'}

# Numeric features only, keep columns with >1 unique value
num_cols = [
    c for c in df_final.select_dtypes(include=[np.number]).columns
    if c not in exclude and df_final[c].nunique(dropna=True) > 1
]

# Variance (spread) and absolute skewness (distribution asymmetry)
variances = df_final[num_cols].var(numeric_only=True).sort_values(ascending=False)
skews = df_final[num_cols].skew(numeric_only=True).abs().sort_values(ascending=False)

# Select top-N features by each metric
TOP_N = 12
top_by_variance = variances.head(TOP_N).index.tolist()
top_by_skew = skews.head(TOP_N).index.tolist()

# Combine into a unique ordered shortlist (variance first, then skew extras)
top_features = list(dict.fromkeys(top_by_variance + top_by_skew))

# Report
print("Top by variance:", top_by_variance)
print("Top by |skew| :", top_by_skew)
print("Combined shortlist:", top_features)


In [ ]:
import numpy as np
import pandas as pd

# === IQR Outlier Summary for ALL numeric columns (df_final) ===

# 1) Pick numeric columns (optionally exclude IDs/meta)
exclude = {'transaction_id', 'property_id'}  # add 'Unnamed: 0','data_join_year' if you want
num_cols = [c for c in df_final.select_dtypes(include=[np.number]).columns
            if c not in exclude and df_final[c].notna().sum() > 0]

rows = []
bounds = {}

for c in num_cols:
    s = df_final[c].dropna()
    n = int(s.size)
    if n < 2:
        rows.append({'column': c, 'n': n, 'q1': np.nan, 'q3': np.nan, 'iqr': np.nan,
                     'lower': np.nan, 'upper': np.nan, 'outlier_count': 0, 'outlier_percent': 0.0})
        bounds[c] = (np.nan, np.nan)
        continue

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    if not np.isfinite(iqr) or iqr == 0:
        lower = upper = np.nan
        out_cnt = 0
    else:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        out_cnt = int(((s < lower) | (s > upper)).sum())

    out_pct = round((out_cnt / n) * 100, 2) if n else 0.0
    rows.append({
        'column': c, 'n': n,
        'q1': q1, 'q3': q3, 'iqr': iqr,
        'lower': lower, 'upper': upper,
        'outlier_count': out_cnt, 'outlier_percent': out_pct
    })
    bounds[c] = (lower, upper)

summary = pd.DataFrame(rows).sort_values('outlier_percent', ascending=False)
print("=== IQR Outlier Summary (all numeric columns) ===")
print(summary[['column','n','outlier_count','outlier_percent','lower','upper']].to_string(index=False))

# --- Optional: inspect actual outlier rows for a chosen column (change 'col' as needed) ---
col = 'lot_sqft'  # <- change to any column from summary['column']
if col in bounds:
    lo, hi = bounds[col]
    if np.isnan(lo) or np.isnan(hi):
        print(f"\n{col}: no IQR bounds (constant/zero IQR).")
    else:
        outliers = df_final[(df_final[col] < lo) | (df_final[col] > hi)]
        print(f"\n{col}: outliers = {len(outliers):,}")
        show_cols = [col, 'sale_price_usd', 'transaction_id', 'property_id']
        show_cols = [c for c in show_cols if c in df_final.columns]
        print(outliers[show_cols].head(20).to_string(index=False))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 11

# 1) Columns to plot: from out_df if available, else fallback list
try:
    outlier_cols = out_df.loc[out_df['outlier_count'] > 0, 'column'].tolist()
except NameError:
    outlier_cols = [
        'lot_sqft','building_grade','sale_price_usd','main_floor_sqft',
        'bedrooms','total_living_sqft','finished_basement_sqft',
        'longitude','full_bathrooms','three_quarter_bathrooms',
        'attached_garage_sqft','home_condition','number_of_stories',
        'half_bathrooms'
    ]

# 2) Keep only existing numeric columns
plot_cols = [
    c for c in outlier_cols
    if c in df_final.columns and pd.api.types.is_numeric_dtype(df_final[c])
]

# 3) Plot boxplots in batches (stacked vertically)
BATCH = 6  # number of boxplots per figure
for i in range(0, len(plot_cols), BATCH):
    batch = plot_cols[i:i+BATCH]
    rows = len(batch)
    fig, axes = plt.subplots(rows, 1, figsize=(12, 2.2 * rows), constrained_layout=True)

    if rows == 1:
        axes = [axes]

    for ax, col in zip(axes, batch):
        sns.boxplot(
            x=df_final[col].dropna(),
            ax=ax,
            color="skyblue",
            linewidth=1,
            fliersize=2  # small dots for outliers
        )
        ax.set_title(col.replace("_", " ").title(), fontsize=11, pad=6)
        ax.set_xlabel("")
        ax.grid(axis='x', linestyle="--", alpha=0.3)

    plt.suptitle("Boxplots of Numeric Features (with Outliers)", fontsize=13, weight="bold", y=1.02)
    plt.show()


In [ ]:
import numpy as np

# Remove leftover meta/index column if present
if 'Unnamed: 0' in df_final.columns:
    df_final.drop(columns=['Unnamed: 0'], inplace=True)

# Columns to cap (keep only those that exist)
cap_cols = [
    'lot_sqft',
    'total_living_sqft',
    'main_floor_sqft',
    'finished_basement_sqft',
    'attached_garage_sqft',
    'basement_garage_sqft',  # only if present
]
cap_cols = [c for c in cap_cols if c in df_final.columns]

# Two-sided winsorization percentiles
q_lo, q_hi = 0.01, 0.99  # adjust to tighten/loosen capping
changed = {}
cutoffs = {}

# Apply capping in-place
for c in cap_cols:
    # Compute cutoffs using available (non-NA) values
    lo, hi = df_final[c].quantile([q_lo, q_hi])
    cutoffs[c] = (lo, hi)

    before = df_final[c].copy()
    df_final[c] = df_final[c].clip(lo, hi)

    # Count how many values were affected
    changed[c] = int((before != df_final[c]).sum())

# Report
print("✅ Outlier capping applied in-place on df_final.")
for c in cap_cols:
    lo, hi = cutoffs[c]
    n = df_final[c].notna().sum()
    pct = (changed[c] / n * 100) if n else 0
    print(f"   {c}: capped {changed[c]:,} values ({pct:.2f}%) "
          f"at [{int(q_lo*100)}–{int(q_hi*100)}] percentiles "
          f"(lo={lo:.2f}, hi={hi:.2f})")


In [ ]:
import matplotlib.pyplot as plt

# pick the capped columns
cap_cols = [
    'lot_sqft',
    'total_living_sqft',
    'main_floor_sqft',
    'finished_basement_sqft',
    'attached_garage_sqft',
    'basement_garage_sqft'
]
cap_cols = [c for c in cap_cols if c in df_final.columns]

# plot histograms after capping
ncols = 2
nrows = (len(cap_cols) + 1) // ncols
plt.figure(figsize=(12, 4*nrows))

for i, col in enumerate(cap_cols, 1):
    plt.subplot(nrows, ncols, i)
    df_final[col].hist(bins=50, edgecolor="black")
    plt.title(f"{col} (after capping)")
    plt.xlabel(col)
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Keep numeric codes for modeling (df_final['property_use_type'] stays as-is)

# Known code → description mapping (extend as needed)
use_map = {
    2:  "Single-family residential",
    6:  "Single-family (C/I zone)",
    29: "Townhouse (multi-family)",
}

# --- Coverage report: which codes exist vs mapped ---
# Coerce to integers safely (handles floats/strings), ignore non-numeric
codes_series = pd.to_numeric(df_final['property_use_type'], errors='coerce')
codes_present = (
    pd.Series(codes_series.dropna().astype(int).unique())
      .sort_values()
      .tolist()
)
unmapped = [c for c in codes_present if c not in use_map]
print("🧩 Codes present:", codes_present)
print("⚠️ Unmapped codes:", unmapped if unmapped else "None")

# --- Create human-readable description with robust fallbacks ---
# 1) map known codes
desc = codes_series.map(use_map)
# 2) explicit label for missing
desc = np.where(df_final['property_use_type'].isna(), "(Missing)", desc)
# 3) anything unmapped becomes Other/Unmapped
df_final['property_use_desc'] = pd.Series(desc, index=df_final.index).fillna("Other/Unmapped")

# Make categorical (for consistent ordering in plots/groupbys)
cat_order = [
    "Single-family residential",
    "Single-family (C/I zone)",
    "Townhouse (multi-family)",
    "Other/Unmapped",
    "(Missing)",
]
df_final['property_use_desc'] = pd.Categorical(
    df_final['property_use_desc'],
    categories=cat_order,
    ordered=False
)

# --- Quick frequency check ---
print("\n📊 Property Use Description:")
print(df_final['property_use_desc'].value_counts(dropna=False))

# --- Plot (Word-ready) ---
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 11

order = [c for c in cat_order if c in df_final['property_use_desc'].unique()]
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)

sns.countplot(
    y='property_use_desc',
    data=df_final,
    order=order,
    edgecolor='black',
    linewidth=0.5,
    ax=ax
)

ax.set_title("Property Use Type Distribution", pad=6, fontsize=12, weight='bold')
ax.set_xlabel("Count")
ax.set_ylabel("")
ax.grid(axis='x', linestyle='--', alpha=0.3)

# Add count (optional percent) labels
counts = df_final['property_use_desc'].value_counts()
total = int(counts.sum())
for p in ax.patches:
    count = int(p.get_width())
    pct = (count / total * 100) if total else 0
    y = p.get_y() + p.get_height()/2
    ax.annotate(f"{count:,}  ({pct:.1f}%)",
                xy=(count, y), xytext=(6, 0), textcoords='offset points',
                va='center', ha='left', fontsize=10)

# Slight right padding so labels don’t clip
xmax = max([p.get_width() for p in ax.patches]) if ax.patches else 0
ax.set_xlim(0, xmax * 1.08 if xmax else 1)

plt.show()


In [ ]:
import pandas as pd

print("🧾 Column Overview:\n")

# Use df_final instead of df
col_info = pd.DataFrame({
    'Data Type': df_final.dtypes,
    'Missing Values': df_final.isnull().sum(),
    'Unique Values': df_final.nunique()
}).sort_index()

print(col_info.to_markdown())


In [ ]:
# === Split into Seattle vs. King County and compare prices ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 11

# --- Safety checks & prep ---
assert 'df_final' in globals(), "df_final not found."

# Ensure is_seattle exists (derive from 'city' if needed)
if 'is_seattle' not in df_final.columns:
    if 'city' in df_final.columns:
        df_final['is_seattle'] = (
            df_final['city'].astype(str).str.strip().str.upper() == 'SEATTLE'
        ).astype('int8')
    else:
        raise ValueError("'is_seattle' not found and 'city' unavailable to derive it.")

# Ensure sale_price_usd is numeric
if 'sale_price_usd' not in df_final.columns:
    raise KeyError("'sale_price_usd' column not found in df_final.")
df_final['sale_price_usd'] = pd.to_numeric(df_final['sale_price_usd'], errors='coerce')

# --- 1) Split by is_seattle ---
df_seattle = df_final.loc[df_final['is_seattle'] == 1].copy()
df_king    = df_final.loc[df_final['is_seattle'] == 0].copy()

print(f"Seattle rows: {len(df_seattle):,}")
print(f"King County rows (excluding Seattle): {len(df_king):,}")

# Guard against empty splits
assert len(df_seattle) > 0, "Seattle split is empty."
assert len(df_king) > 0, "King County split is empty."

# --- 2) Drop only helper/meta columns in each split ---
def drop_helpers(df):
    to_drop_exact = ['is_seattle', 'Unnamed: 0']    # meta/helpers if present
    to_drop_prefixes = ['_data_source']             # provenance columns
    cols_drop = set(c for c in to_drop_exact if c in df.columns)
    cols_drop |= {c for c in df.columns if any(c.startswith(p) for p in to_drop_prefixes)}
    if cols_drop:
        df.drop(columns=sorted(cols_drop), inplace=True)
    return sorted(cols_drop)

dropped_sea = drop_helpers(df_seattle)
dropped_kc  = drop_helpers(df_king)
print(f"\n🧹 Dropped in Seattle: {dropped_sea if dropped_sea else 'None'}")
print(f"🧹 Dropped in King   : {dropped_kc if dropped_kc else 'None'}")

# --- 3) Peek columns/rows ---
pd.set_option('display.max_columns', None)
print("\n📌 Seattle dataset columns:", df_seattle.columns.tolist())
print("\n📌 King County dataset columns:", df_king.columns.tolist())
print("\n🔎 Seattle head(3):"); print(df_seattle.head(3))
print("\n🔎 King County head(3):"); print(df_king.head(3))

# --- 4) Median sale price (side-by-side) ---
med_sea = float(np.nanmedian(df_seattle['sale_price_usd'].values))
med_kc  = float(np.nanmedian(df_king['sale_price_usd'].values))
print(f"\n💰 Median Sale Price — Seattle:     ${med_sea:,.0f}")
print(f"💰 Median Sale Price — King County: ${med_kc:,.0f}")

# --- 5) Sample size + price distributions (log x-axis, safe for nonpositive) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# (a) Sample size bars with labels
ax = axes[0]
bars = ax.bar(['Seattle', 'King County'], [len(df_seattle), len(df_king)])
ax.set_ylabel("Number of Properties")
ax.set_title("Sample Size by Region")
for b in bars:
    ax.annotate(f"{int(b.get_height()):,}", (b.get_x() + b.get_width()/2, b.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)

# (b) Price distributions (log scale); filter out nonpositive/NaN safely
ax = axes[1]
sea_prices = df_seattle['sale_price_usd'].dropna()
kc_prices  = df_king['sale_price_usd'].dropna()
sea_prices = sea_prices[sea_prices > 0]
kc_prices  = kc_prices[kc_prices > 0]

ax.hist(sea_prices, bins=60, alpha=0.6, density=True, label='Seattle')
ax.hist(kc_prices,  bins=60, alpha=0.6, density=True, label='King County')

# Median lines
ax.axvline(med_sea, linestyle='--', linewidth=2, label=f"Seattle median ${med_sea:,.0f}")
ax.axvline(med_kc,  linestyle='--', linewidth=2, label=f"King median ${med_kc:,.0f}")

# Log scale (guarded)
if (len(sea_prices) > 0 or len(kc_prices) > 0) and min(sea_prices.min() if len(sea_prices) else np.inf,
                                                        kc_prices.min() if len(kc_prices) else np.inf) > 0:
    ax.set_xscale('log')

ax.set_xlabel("Sale Price (USD, log scale)")
ax.set_ylabel("Density")
ax.set_title("Price Distribution")
ax.legend()

plt.show()


In [ ]:
import numpy as np
import pandas as pd

def fe_minimal(df, add_distances=True, drop_view_parts=True, drop_garage_parts=True):
    """
    Minimal feature engineering for housing data.

    Parameters
    ----------
    df : DataFrame
        Input data (will NOT be modified in-place).
    add_distances : bool
        If True, adds Haversine distances to key POIs (requires latitude/longitude).
    drop_view_parts : bool
        If True, drops original view_* columns after aggregations.
    drop_garage_parts : bool
        If True, drops garage part columns after creating total_garage_sqft.

    Returns
    -------
    DataFrame
        Copy of df with engineered features added and optional columns dropped.
    """
    df = df.copy()
    created, dropped = [], []

    def to_num(cols):
        """Coerce listed columns to numeric (safe, NaN on non-numeric)."""
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors='coerce')

    # --- Time features (sale year/month; ages) ---
    if 'sale_date' in df.columns:
        df['sale_date'] = pd.to_datetime(df['sale_date'], errors='coerce')
        df['sale_year']  = df['sale_date'].dt.year
        df['sale_month'] = df['sale_date'].dt.month
        created += ['sale_year', 'sale_month']

    if {'sale_year', 'year_built'}.issubset(df.columns):
        to_num(['year_built'])
        df['age_at_sale'] = (df['sale_year'] - df['year_built']).clip(lower=0).astype('float32')
        created.append('age_at_sale')

    if {'sale_year', 'year_renovated', 'year_built'}.issubset(df.columns):
        to_num(['year_renovated', 'year_built'])
        valid_reno = df['year_renovated'].notna() & (df['year_renovated'] > df['year_built'])
        yrs = (df['sale_year'] - df['year_renovated']).where(valid_reno, 0)
        df['yrs_since_reno'] = yrs.clip(lower=0).astype('float32')
        created.append('yrs_since_reno')

    # --- Bathrooms (keep as-is; remove any prior weighted total) ---
    to_num(['total_living_sqft', 'full_bathrooms', 'three_quarter_bathrooms', 'half_bathrooms'])
    if 'total_baths_weighted' in df.columns:
        df.drop(columns=['total_baths_weighted'], inplace=True); dropped.append('total_baths_weighted')

    # --- Garage: total_garage_sqft (+ optional drop of parts) ---
    parts = [c for c in ['attached_garage_sqft', 'basement_garage_sqft'] if c in df.columns]
    to_num(parts)
    if parts:
        a = df['attached_garage_sqft'] if 'attached_garage_sqft' in df.columns else pd.Series(0, index=df.index)
        b = df['basement_garage_sqft'] if 'basement_garage_sqft' in df.columns else pd.Series(0, index=df.index)
        df['total_garage_sqft'] = (a.fillna(0) + b.fillna(0)).astype('float32')
        created.append('total_garage_sqft')
        if drop_garage_parts:
            df.drop(columns=[c for c in parts if c in df.columns], inplace=True); dropped += parts

    # --- Views: compact aggregations (+ optional drop of parts) ---
    all_view_cols = [c for c in df.columns if c.startswith('view_')]
    water = [c for c in ['view_puget_sound', 'view_lake_washington', 'view_lake_sammamish', 'view_other_water'] if c in df.columns]
    mtn   = [c for c in ['view_mountain_rainier', 'view_olympic_mountains', 'view_cascade_mountains'] if c in df.columns]
    skyline_src = 'view_seattle_skyline' if 'view_seattle_skyline' in df.columns else None

    if water:
        df['any_water_view'] = df[water].fillna(0).astype(int).max(axis=1).astype('int8'); created.append('any_water_view')
    if mtn:
        df['any_mountain_view'] = df[mtn].fillna(0).astype(int).max(axis=1).astype('int8'); created.append('any_mountain_view')
    if skyline_src:
        df['skyline_view'] = df[skyline_src].fillna(0).astype(int).astype('int8'); created.append('skyline_view')
    if all_view_cols:
        df['view_count'] = df[all_view_cols].fillna(0).astype(int).sum(axis=1).astype('int16'); created.append('view_count')
    if drop_view_parts and all_view_cols:
        df.drop(columns=all_view_cols, inplace=True); dropped += all_view_cols

    # --- Distances to key POIs (Haversine; km) ---
    if add_distances and {'latitude', 'longitude'}.issubset(df.columns):
        to_num(['latitude', 'longitude'])
        valid = df['latitude'].notna() & df['longitude'].notna()

        def haversine_km(lat1, lon1, lat2, lon2):
            lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
            dlat, dlon = lat2 - lat1, lon2 - lon1
            a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
            return 6371.0088 * (2*np.arcsin(np.sqrt(a)))

        pois = {
            'downtown_core' : (47.6062, -122.3321),
            'seatac_airport': (47.4502, -122.3088),
            'bellevue_dt'   : (47.6101, -122.2015),
            'redmond_dt'    : (47.6730, -122.1215),
        }
        for name, (plat, plon) in pois.items():
            col = f'dist_km_{name}'
            df[col] = np.nan
            if valid.any():
                df.loc[valid, col] = haversine_km(
                    df.loc[valid, 'latitude'].values, df.loc[valid, 'longitude'].values, plat, plon
                ).astype('float32')
            created.append(col)

    # --- Meta cleanup (remove helper/provenance columns if present) ---
    for c in ['property_use_desc', 'Unnamed: 0']:
        if c in df.columns:
            df.drop(columns=[c], inplace=True); dropped.append(c)
    junk = [c for c in df.columns if c.startswith('_data_source')]
    if junk:
        df.drop(columns=junk, inplace=True); dropped += junk

    # --- Report summary ---
    print("✅ Created:", created if created else "none")
    print("🗑️  Dropped:", dropped if dropped else "none")
    print("📐 Shape:", df.shape)

    return df

# Re-run minimal FE on both splits (unchanged behavior)
df_seattle = fe_minimal(df_seattle, add_distances=True)
df_king    = fe_minimal(df_king,    add_distances=True)


In [ ]:
# === Split-only pipeline (no capping/duplicates/leakage) ===
# - Diagnose skew per split
# - Pick common features to log
# - Log those features IN PLACE (overwrite raw)
# - Log the target IN PLACE: overwrite 'sale_price_usd'
# - Show before/after histograms (temporary copies only)
# - (Optional) align columns by union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 1) Skew diagnostics per region
def skew_report_by_region(df, region_name, skew_thresh=1.0, show_top=20, extra_dont_log=None):
    """
    Compute skewness for all numeric columns and suggest log1p candidates.
    Excludes 0/1 binaries, negative-valued columns, and date/ID/meta fields.
    Returns: {'report': DataFrame, 'to_log_features': list, 'target_skew': float|None}
    """
    assert isinstance(df, pd.DataFrame), f"{region_name}: df must be a DataFrame"

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        print(f"{region_name}: No numeric columns found.")
        return None

    default_dont_log = [
        'sale_year', 'sale_month', 'data_join_year',
        'transaction_id', 'property_id', 'sale_id', 'pinx'
    ]
    dont_log = set(c for c in default_dont_log if c in df.columns)
    if extra_dont_log:
        dont_log |= set(c for c in extra_dont_log if c in df.columns)

    def is_binary(series: pd.Series) -> bool:
        vals = pd.Series(series.dropna().unique())
        return len(vals) > 0 and vals.isin([0, 1]).all()

    binary_cols = [c for c in num_cols if is_binary(df[c])]

    sk = df[num_cols].skew(numeric_only=True)
    rep = sk.reset_index()
    rep.columns = ['column', 'skewness']
    rep['abs_skew'] = rep['skewness'].abs()

    nonneg = {c: (df[c].min(skipna=True) >= 0) for c in num_cols}
    rep['is_binary'] = rep['column'].isin(binary_cols)
    rep['non_negative'] = rep['column'].map(nonneg)
    rep['dont_log'] = rep['column'].isin(dont_log)

    rep['log_candidate'] = (
        (rep['abs_skew'] > skew_thresh) &
        (~rep['is_binary']) &
        (rep['non_negative']) &
        (~rep['dont_log'])
    )

    rep = rep.sort_values('abs_skew', ascending=False)

    target_skew = None
    if 'sale_price_usd' in rep['column'].values:
        target_skew = float(rep.loc[rep['column'] == 'sale_price_usd', 'skewness'].iloc[0])

    print(f"\n=== Skewness Report — {region_name} ===")
    if target_skew is not None:
        print(f"Target (sale_price_usd) skewness: {target_skew:,.3f}")
    print("\nTop skewed numeric columns:")
    display(rep.head(show_top))

    to_log = rep.loc[rep['log_candidate'], 'column'].tolist()
    to_log_features = [c for c in to_log if c != 'sale_price_usd']  # target handled separately

    print(f"\nSuggested features to log1p (exclude binary/negatives) [{len(to_log_features)}]:")
    print(to_log_features)

    return {'report': rep, 'to_log_features': to_log_features, 'target_skew': target_skew}

# 2) Run diagnostics on splits
sea_skew = skew_report_by_region(df_seattle, "Seattle", skew_thresh=1.0, show_top=20)
kc_skew  = skew_report_by_region(df_king,    "King County", skew_thresh=1.0, show_top=20)

# 3) Common set to log in BOTH splits
common_to_log = []
if sea_skew and kc_skew:
    common_to_log = sorted(set(sea_skew['to_log_features']).intersection(kc_skew['to_log_features']))
print("\n✅ Common features to log1p across both splits:")
print(common_to_log)

# 4) Save BEFORE copies for images (temporary; no new columns added)
sea_before = df_seattle.copy()
kc_before  = df_king.copy()

# 5) Log features IN PLACE (overwrite; no *_log duplicates)
def apply_logs_overwrite(df, cols):
    applied = []
    for c in cols:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors='coerce')
            if s.min(skipna=True) >= 0:
                df[c] = np.log1p(s)
                applied.append(c)
            else:
                print(f"Skipped '{c}' (has negatives).")
    return applied

sea_transformed = apply_logs_overwrite(df_seattle, common_to_log)
kc_transformed  = apply_logs_overwrite(df_king,    common_to_log)
print("\n✅ Logged (overwrote) in Seattle:", sea_transformed)
print("✅ Logged (overwrote) in King County:", kc_transformed)

# 6) Log the TARGET IN PLACE — overwrite 'sale_price_usd'
def log_target_inplace(df, target='sale_price_usd'):
    if target not in df.columns:
        print(f"Target '{target}' not found; skipping.")
        return False
    s = pd.to_numeric(df[target], errors='coerce')
    neg_count = int((s < 0).sum())
    if neg_count > 0:
        print(f"Warning: {neg_count} negative target values clipped to 0 before log1p.")
    df[target] = np.log1p(s.clip(lower=0))
    return True

sea_ok = log_target_inplace(df_seattle, target='sale_price_usd')
kc_ok  = log_target_inplace(df_king,    target='sale_price_usd')
print("\n✅ Target transformed IN PLACE: 'sale_price_usd' overwritten with log1p in both splits (no duplicates, nothing dropped).")

# 7) Images — before vs after histograms (uses temporary copies only)
def plot_before_after(df_before, df_after, cols, title_prefix, transformed_cols, show_target=True):
    transformed_cols = set(transformed_cols)
    cols = [c for c in cols if c in df_before.columns and c in df_after.columns]
    for col in cols:
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        df_before[col].hist(bins=60)
        plt.title(f"{title_prefix} — {col} (before/raw)")
        plt.xlabel(col); plt.ylabel("Frequency")

        plt.subplot(1, 2, 2)
        df_after[col].hist(bins=60)
        if col in transformed_cols:
            plt.title(f"{title_prefix} — {col} (after, log1p applied)")
        else:
            plt.title(f"{title_prefix} — {col} (after, unchanged)")
        plt.xlabel(col); plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()

    # Target (overwritten in place): compare before vs after
    if show_target and 'sale_price_usd' in df_before.columns and 'sale_price_usd' in df_after.columns:
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        df_before['sale_price_usd'].hist(bins=60)
        plt.title(f"{title_prefix} — sale_price_usd (before/raw)")
        plt.xlabel('sale_price_usd'); plt.ylabel("Frequency")

        plt.subplot(1, 2, 2)
        df_after['sale_price_usd'].hist(bins=60)
        plt.title(f"{title_prefix} — sale_price_usd (after, log1p)")
        plt.xlabel('sale_price_usd'); plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()

to_show = ['lot_sqft', 'total_living_sqft']  # adjust as needed
plot_before_after(sea_before, df_seattle, to_show, "Seattle", transformed_cols=sea_transformed, show_target=True)
plot_before_after(kc_before,  df_king,    to_show, "King County", transformed_cols=kc_transformed, show_target=True)

# 8) (Optional) align columns by UNION (fills missing with NaN)
# all_cols = sorted(set(df_seattle.columns) | set(df_king.columns))
# df_seattle = df_seattle.reindex(columns=all_cols)
# df_king    = df_king.reindex(columns=all_cols)
# print("\n✅ Aligned columns to UNION. Shapes:", df_seattle.shape, df_king.shape)

# 9) Summary
print("\n=== SUMMARY ===")
print(f"Features logged in BOTH splits (overwrote raw): {common_to_log}")
print("Target transformed IN PLACE: 'sale_price_usd' is now log-scaled (no duplicate column, nothing dropped).")
print("When modeling: inverse-transform predictions with np.expm1 to return to price units.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_correlation_heatmap(df, title_prefix, target='sale_price_usd_log', top_k=30):
    # numeric only
    num = df.select_dtypes(include=['number']).copy()
    if num.empty:
        print(f"{title_prefix}: no numeric columns to correlate.")
        return

    # drop constant columns
    nunique = num.nunique(dropna=False)
    num = num.loc[:, nunique > 1]
    if num.empty:
        print(f"{title_prefix}: all numeric columns are constant.")
        return

    # choose columns: top_k by |corr with target| if target exists
    if target in num.columns:
        corr_full = num.corr(numeric_only=True)
        target_abs = corr_full[target].dropna().abs().sort_values(ascending=False)
        keep = target_abs.index[:top_k].tolist()
        # put target first
        cols = [target] + [c for c in keep if c != target]
        corr = num[cols].corr(numeric_only=True)
        title = f"{title_prefix} – Correlation Heatmap vs {target} (top {len(cols)})"
    else:
        # fallback: full corr on all numeric columns
        corr = num.corr(numeric_only=True)
        title = f"{title_prefix} – Correlation Heatmap (numeric only)"

    plt.figure(figsize=(14, 11))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, cbar=True)
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Seattle heatmap
plot_correlation_heatmap(df_seattle, "Seattle")

# King County heatmap
plot_correlation_heatmap(df_king, "King County")


In [ ]:
# === Encoding → Drop raw text → Target log (in-place) → Bool→int8 → Audit ===
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from pandas.api.types import (
    is_string_dtype, is_object_dtype, is_numeric_dtype, is_bool_dtype
)

# ---------------- CONFIG ----------------
LOW_CARD_MAX  = 15
MID_CARD_MAX  = 50
FORCE_FREQ    = {'city', 'submarket_area'}
FORCE_ONEHOT  = set()
TOPK_ONEHOT   = 20

# Exclusions (IDs/meta + target) — keep target out of encoding
EXCLUDE_CATS = {
    'transaction_id','property_id','sale_id','pinx',
    'sale_price_usd'
}

DROP_SALE_WARNING = False
TARGET_COL = 'sale_price_usd'  # single canonical target (log1p IN PLACE)

# ---------------- Helpers ----------------
def _is_string_or_categorical_dtype(s: pd.Series) -> bool:
    return (
        is_string_dtype(s.dtype)
        or is_object_dtype(s.dtype)
        or isinstance(s.dtype, pd.CategoricalDtype)
    )

def _clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype('string').str.strip().str.lower()
    return s.fillna('missing')

def _one_hot_topk(series: pd.Series, name: str, topk: int) -> pd.DataFrame:
    top_vals = series.value_counts().index[:topk]
    trimmed = series.where(series.isin(top_vals), other='other')
    return pd.get_dummies(trimmed, prefix=name, dummy_na=False)

def encode_categoricals(df: pd.DataFrame,
                        low_card_max=LOW_CARD_MAX,
                        mid_card_max=MID_CARD_MAX,
                        force_freq=FORCE_FREQ,
                        force_onehot=FORCE_ONEHOT,
                        topk_onehot=TOPK_ONEHOT,
                        drop_raw=True,
                        drop_sale_warning=DROP_SALE_WARNING):
    df_enc = df.copy()

    if drop_sale_warning and 'sale_warning_code' in df_enc.columns:
        df_enc.drop(columns=['sale_warning_code'], inplace=True, errors='ignore')

    # 1) detect text-like columns (exclude IDs/target)
    cat_cols = [c for c in df_enc.columns if c not in EXCLUDE_CATS and _is_string_or_categorical_dtype(df_enc[c])]
    if not cat_cols:
        print("No categorical columns detected.")
        return df_enc, {"one_hot_cols":[], "ordinal_cols":[], "freq_cols":[], "ordinal_maps":{}, "added_cols":[]}

    for c in cat_cols:
        df_enc[c] = _clean_text_series(df_enc[c])

    # 2) cardinality split
    card = {c: int(df_enc[c].nunique(dropna=True)) for c in cat_cols}
    one_hot_cols, ordinal_cols, freq_cols = [], [], []
    for c in cat_cols:
        if c in force_onehot:
            one_hot_cols.append(c)
        elif c in force_freq:
            freq_cols.append(c)
        else:
            k = card[c]
            if k <= low_card_max:
                one_hot_cols.append(c)
            elif k <= mid_card_max:
                ordinal_cols.append(c)
            else:
                freq_cols.append(c)

    added_cols = []

    # 3) one-hot
    if one_hot_cols:
        blocks = []
        for c in one_hot_cols:
            if c in force_onehot and topk_onehot is not None:
                dummies = _one_hot_topk(df_enc[c], c, topk_onehot)
            else:
                dummies = pd.get_dummies(df_enc[c], prefix=c, dummy_na=False)
            blocks.append(dummies)
            added_cols.extend(dummies.columns.tolist())
        df_enc = pd.concat([df_enc] + blocks, axis=1)

    # 4) ordinal
    ordinal_maps = {}
    if ordinal_cols:
        oe = OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
            encoded_missing_value=-1,
            dtype=np.int32
        )
        enc_vals = oe.fit_transform(df_enc[ordinal_cols])
        enc_df = pd.DataFrame(enc_vals, columns=[f"{c}_ord" for c in ordinal_cols], index=df_enc.index)
        df_enc = pd.concat([df_enc, enc_df], axis=1)
        added_cols.extend(enc_df.columns.tolist())
        for i, c in enumerate(ordinal_cols):
            cats = list(oe.categories_[i])
            ordinal_maps[c] = {cat: idx for idx, cat in enumerate(cats)}

    # 5) frequency
    for c in freq_cols:
        freq_map = df_enc[c].value_counts(normalize=True, dropna=False).to_dict()
        colname = f"{c}_freq"
        df_enc[colname] = df_enc[c].map(freq_map).astype('float32')
        added_cols.append(colname)

    # 6) drop original text columns (numeric-only ready)
    if drop_raw and cat_cols:
        df_enc.drop(columns=cat_cols, inplace=True, errors='ignore')

    meta = {
        "one_hot_cols": one_hot_cols,
        "ordinal_cols": ordinal_cols,
        "freq_cols":    freq_cols,
        "ordinal_maps": ordinal_maps,
        "added_cols":   added_cols
    }

    print(f"\n✅ Encoding complete.")
    print(f"  • One-hot: {one_hot_cols}")
    print(f"  • Ordinal: {ordinal_cols}")
    print(f"  • Frequency: {freq_cols}")
    print(f"  • Added {len(added_cols)} columns. New shape: {df_enc.shape}")

    return df_enc, meta

def log_target_inplace(df: pd.DataFrame, col: str = TARGET_COL) -> pd.DataFrame:
    """Log-transform the target in place if it still looks raw."""
    out = df.copy()
    if col not in out.columns:
        raise ValueError(f"Missing '{col}' in dataframe.")
    s = pd.to_numeric(out[col], errors='coerce')
    if s.max(skipna=True) > 50:  # raw prices >> 50; logged ~10–16
        out[col] = np.log1p(s.clip(lower=0)).astype('float32')
    return out

def make_numeric_only(df: pd.DataFrame):
    out = df.copy()
    bool_cols = [c for c in out.columns if is_bool_dtype(out[c])]
    if bool_cols:
        out[bool_cols] = out[bool_cols].astype('int8')
    non_numeric = [c for c in out.columns if not is_numeric_dtype(out[c])]
    return out, bool_cols, non_numeric

# ---------------- APPLY TO BOTH REGIONS ----------------
assert 'df_seattle' in globals() and 'df_king' in globals(), "Run your split first (df_seattle, df_king)."

# 1) Encode & drop raw text
df_seattle_enc, enc_meta_sea = encode_categoricals(df_seattle, drop_raw=True)
df_king_enc,    enc_meta_kc  = encode_categoricals(df_king,    drop_raw=True)

# 2) Target: log in place on sale_price_usd only
df_seattle_enc = log_target_inplace(df_seattle_enc, col=TARGET_COL)
df_king_enc    = log_target_inplace(df_king_enc,    col=TARGET_COL)

# 3) Make numeric-only (bool→int8) + audit
df_seattle_enc, sea_bool_cols, sea_nonnum = make_numeric_only(df_seattle_enc)
df_king_enc,    kc_bool_cols,  kc_nonnum  = make_numeric_only(df_king_enc)

print("\n=== Final Shapes (numeric-ready) ===")
print("Seattle (encoded):     ", df_seattle_enc.shape)
print("King County (encoded): ", df_king_enc.shape)

print("\nSeattle — converted bool→int8:", sea_bool_cols if sea_bool_cols else "none")
print("Seattle — non-numeric remaining:", sea_nonnum if sea_nonnum else "none")

print("\nKing — converted bool→int8:", kc_bool_cols if kc_bool_cols else "none")
print("King — non-numeric remaining:", kc_nonnum if kc_nonnum else "none")

# 4) Quick numeric-only sanity
sea_num_only = [c for c in df_seattle_enc.columns if is_numeric_dtype(df_seattle_enc[c])]
kc_num_only  = [c for c in df_king_enc.columns    if is_numeric_dtype(df_king_enc[c])]
print(f"\nNumeric features (Seattle): {len(sea_num_only)} / {df_seattle_enc.shape[1]}")
print(f"Numeric features (King)   : {len(kc_num_only)} / {df_king_enc.shape[1]}")


In [ ]:
def show_cols_and_head(name, df):
    print(f"\n=== {name} Encoded Columns (count) === {df.shape[1]}")
    print(list(df.columns))
    print(f"\n=== {name} head(3) ===")
    print(df.head(3).to_string(index=False))

# Seattle
show_cols_and_head("Seattle", df_seattle_enc)

# King County
show_cols_and_head("King County", df_king_enc)


In [ ]:
# ===== Seattle — 5-Fold CV, Scatter Plots (grid only), Feature Counts + MAPE =====
# Requires: df_seattle_enc exists and TARGET 'sale_price_usd' is log1p in-place.

import os, numpy as np, pandas as pd, xgboost as xgb, matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from matplotlib.ticker import FuncFormatter

# -------------------- CONFIG --------------------
CONFIG = {
    'cv_folds': 5,
    'params': {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse',
        'eta': 0.093, 'max_depth': 7, 'min_child_weight': 5,
        'subsample': 0.609, 'colsample_bytree': 0.983,
        'reg_alpha': 0.145, 'reg_lambda': 1.283, 'gamma': 0.033,
        'tree_method': 'hist', 'grow_policy': 'lossguide', 'seed': 42,
    },
    'num_boost_round': 4453,
    'early_stopping_rounds': 150,
    'plots_dir': 'plots',
    'dpi': 220,
    'topn_fi': 30,
    'top_scatter_feats': 3   # top features for scatter grid
}
os.makedirs(CONFIG['plots_dir'], exist_ok=True)
plt.rcParams['font.size'] = 11

TARGET = "sale_price_usd"  # already log1p upstream

PHYSICAL_FEATURES = [
    'year_built','year_renovated','lot_sqft',
    'total_living_sqft','main_floor_sqft','finished_basement_sqft','total_garage_sqft',
    'building_grade','basement_grade','home_condition',
    'number_of_stories','bedrooms',
    'full_bathrooms','three_quarter_bathrooms','half_bathrooms'
]

# -------------------- HELPERS --------------------
def money_short(x):
    x = float(x)
    return f"${x/1e6:.1f}M" if x>=1e6 else (f"${x/1e3:,.0f}k" if x>=1e3 else f"${x:,.0f}")

def _money_axis():
    return FuncFormatter(lambda t,_: f'${t/1e6:.1f}M' if t>=1e6 else f'${t/1e3:,.0f}k')

def select_numeric_full(df: pd.DataFrame):
    num_df = df.select_dtypes(include=['number','bool']).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors='coerce')
    def is_price_leak(col: str):
        lc = col.lower()
        return col != TARGET and (lc.startswith('sale_price') or lc.endswith('_price') or lc.endswith('price_usd') or lc=='log_price')
    feats = [c for c in num_df.columns if c != TARGET and not is_price_leak(c)]
    return num_df[feats].astype('float32'), num_df[TARGET].astype(float).values, feats

def select_numeric_physical(df: pd.DataFrame):
    num_df = df.select_dtypes(include=['number','bool']).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors='coerce')
    feats = [c for c in PHYSICAL_FEATURES if c in num_df.columns]
    return num_df[feats].astype('float32'), num_df[TARGET].astype(float).values, feats

def rmse_from_mse(mse):
    return float(np.sqrt(mse))

def run_cv_oof(X: pd.DataFrame, y: np.ndarray, label: str):
    kf = KFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=42)
    metrics, best_iters = [], []
    feat_names = list(map(str, X.columns))
    oof_pred = np.full_like(y, np.nan, dtype=float)
    for fold, (tr, va) in enumerate(kf.split(X), 1):
        print(f"\n{label} — Fold {fold}/{CONFIG['cv_folds']}")
        dtr = xgb.DMatrix(X.iloc[tr].values, label=y[tr], feature_names=feat_names)
        dva = xgb.DMatrix(X.iloc[va].values, label=y[va], feature_names=feat_names)
        evals_result = {}
        bst = xgb.train(
            params=CONFIG['params'],
            dtrain=dtr,
            num_boost_round=CONFIG['num_boost_round'],
            evals=[(dtr,'train'), (dva,'val')],
            early_stopping_rounds=CONFIG['early_stopping_rounds'],
            evals_result=evals_result,
            verbose_eval=False
        )
        v = np.array(evals_result['val']['rmse'])
        best_iter = int(np.argmin(v)) + 1
        best_iters.append(best_iter)
        try:
            y_pred_log = bst.predict(dva, iteration_range=(0, best_iter))
        except TypeError:
            y_pred_log = bst.predict(dva, ntree_limit=best_iter)

        rmse_log = rmse_from_mse(mean_squared_error(y[va], y_pred_log))
        mae_log  = float(mean_absolute_error(y[va], y_pred_log))
        r2_log   = float(r2_score(y[va], y_pred_log))
        y_true, y_pred = np.expm1(y[va]), np.expm1(y_pred_log)
        rmse_usd = rmse_from_mse(mean_squared_error(y_true, y_pred))
        mae_usd  = float(mean_absolute_error(y_true, y_pred))
        mape     = float(np.mean(np.abs((y_true - y_pred)/np.clip(y_true,1,None)))*100)

        print(f"  Fold {fold}: RMSE(log)={rmse_log:.3f}  MAE(log)={mae_log:.3f}  "
              f"R²={r2_log:.2f}  RMSE($)={money_short(rmse_usd)}  "
              f"MAE($)={money_short(mae_usd)}  MAPE={mape:.1f}%")

        metrics.append(dict(rmse_log=rmse_log, mae_log=mae_log, r2_log=r2_log,
                            rmse_usd=rmse_usd, mae_usd=mae_usd, mape=mape))
        oof_pred[va] = y_pred_log

    met_df = pd.DataFrame(metrics)
    return met_df, best_iters, y.astype(float), oof_pred

def print_cv_results(name: str, met: pd.DataFrame, out_csv_path: str):
    print(f"\n{name} — {CONFIG['cv_folds']}-FOLD RESULTS")
    for i, row in met.iterrows():
        print(f"  Fold {i+1}: RMSE(log)={row['rmse_log']:.3f}  MAE(log)={row['mae_log']:.3f}  "
              f"R²={row['r2_log']:.2f}  RMSE($)={money_short(row['rmse_usd'])}  "
              f"MAE($)={money_short(row['mae_usd'])}  MAPE={row['mape']:.1f}%")
    avg = met.mean(numeric_only=True)
    print(f"\n{name} — AVERAGE")
    print(f"  RMSE(log)={avg['rmse_log']:.3f}  MAE(log)={avg['mae_log']:.3f}  "
          f"R²={avg['r2_log']:.2f}  RMSE($)={money_short(avg['rmse_usd'])}  "
          f"MAE($)={money_short(avg['mae_usd'])}  MAPE={avg['mape']:.1f}%")
    met.assign(fold=np.arange(1, len(met)+1)).to_csv(out_csv_path, index=False)
    print(f"Saved CV table: {out_csv_path}")

def plot_pred_vs_actual(y_true_log, y_pred_log, title, out_path):
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    rmse = rmse_from_mse(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    ymax = np.nanquantile(np.concatenate([y_true, y_pred]), 0.995)

    plt.figure(figsize=(7.0, 6.2), constrained_layout=True)
    plt.scatter(y_true, y_pred, s=8, alpha=0.35, color='steelblue')
    plt.plot([0, ymax], [0, ymax], 'k--', lw=1)
    plt.xlim(0, ymax); plt.ylim(0, ymax)
    ax = plt.gca()
    ax.xaxis.set_major_formatter(_money_axis())
    ax.yaxis.set_major_formatter(_money_axis())
    plt.xlabel("Actual Price (USD)", fontsize=12)
    plt.ylabel("Predicted Price (USD)", fontsize=12)
    # ----- Title updated to clearly state the graph -----
    plt.title("Seattle (FULL) — Scatter Plot", fontsize=13)
    # ----------------------------------------------------
    plt.grid(True, alpha=0.3)
    plt.savefig(out_path, dpi=CONFIG['dpi'], bbox_inches='tight')
    plt.show()
    plt.close()

def train_full_and_importance(X: pd.DataFrame, y: np.ndarray, rounds: int):
    feat_names = list(map(str, X.columns))
    bst = xgb.train(params=CONFIG['params'],
                    dtrain=xgb.DMatrix(X.values, label=y, feature_names=feat_names),
                    num_boost_round=rounds, verbose_eval=False)
    g = pd.Series(bst.get_score(importance_type='gain'), dtype=float)
    imp = g.reindex(feat_names).fillna(0.0).reset_index()
    imp.columns = ['feature','gain']
    return bst, imp.sort_values('gain', ascending=False)

def plot_feature_importance(imp_df, title, top_n=10, out_path=None):
    top_features = imp_df.head(top_n).copy()
    top_features = top_features.sort_values('gain', ascending=True)

    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(top_features)))
    plt.barh(range(len(top_features)), top_features['gain'], color=colors, alpha=0.7)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance (Gain)', fontsize=12)
    plt.title(title, fontsize=14, pad=20)
    plt.grid(True, alpha=0.3, axis='x')
    for i, v in enumerate(top_features['gain']):
        plt.text(v + max(top_features['gain'])*0.01, i, f'{v:.3f}', va='center', fontsize=10)
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=CONFIG['dpi'], bbox_inches='tight')
    plt.show()
    plt.close()

def plot_feature_scatter_grid(X, y_true_log, features, out_path, max_plots=3):
    y_true = np.expm1(y_true_log)
    feats = [f for f in features if f in X.columns][:max_plots]
    if not feats:
        return

    cols = 3 if len(feats) == 3 else 2
    rows = int(np.ceil(len(feats)/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4.8, rows*4.2), constrained_layout=True)
    axes = np.ravel(axes)

    for ax, f in zip(axes, feats):
        x_arr = pd.to_numeric(X[f], errors='coerce').to_numpy(dtype=float)
        ok = np.isfinite(x_arr) & np.isfinite(y_true)
        ax.scatter(x_arr[ok], y_true[ok], s=8, alpha=0.30, color='steelblue')
        ax.set_title(f, fontsize=11)
        ax.set_xlabel(f)
        ax.yaxis.set_major_formatter(_money_axis())
        ax.set_ylabel("Price (USD)")
        ax.grid(True, alpha=0.3)

    for j in range(len(feats), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Top Features vs Actual Price (USD)", y=1.02, fontsize=12)
    plt.savefig(out_path, dpi=CONFIG['dpi'], bbox_inches='tight')
    plt.show()
    plt.close()

def display_feature_importance(imp_df, top_n=10):
    print(f"\nTop {top_n} Features by Gain:")
    print("-" * 60)
    for i, row in imp_df.head(top_n).iterrows():
        print(f"{i+1:2d}. {row['feature']:30s} {row['gain']:10.4f}")
    print("-" * 60)

# -------------------- RUN --------------------
assert 'df_seattle_enc' in globals(), "df_seattle_enc not found."
assert TARGET in df_seattle_enc.columns, f"Missing {TARGET} in df_seattle_enc."
dfS = df_seattle_enc.copy()

# FULL
X_full, y, full_feats = select_numeric_full(dfS)
print(f"Seattle (FULL): {len(full_feats)} features used")
met_full, best_iters_full, oof_true_full, oof_pred_full = run_cv_oof(X_full, y, "Seattle (FULL)")
print_cv_results("Seattle (FULL)", met_full, os.path.join(CONFIG['plots_dir'], "seattle_full_cv_results.csv"))
median_rounds = int(np.median(best_iters_full)) if len(best_iters_full) else CONFIG['num_boost_round']
best_rounds_full = max(100, int(round(0.5*CONFIG['num_boost_round'] + 0.5*median_rounds)))
bst_full, imp_full = train_full_and_importance(X_full, y, best_rounds_full)

display_feature_importance(imp_full, 20)
plot_feature_importance(imp_full, "Seattle (FULL) — Top 20 Feature Importance",
                        top_n=20,
                        out_path=os.path.join(CONFIG['plots_dir'], "04_seattle_full_feature_importance.png"))

plot_pred_vs_actual(oof_true_full, oof_pred_full,
                    "Seattle (FULL) — Out-of-Fold Predictions vs Actual",
                    os.path.join(CONFIG['plots_dir'], "01_seattle_full_pred_vs_actual.png"))

# ONE scatter grid only
top_feats_full = imp_full['feature'].head(CONFIG['top_scatter_feats']).tolist()
plot_feature_scatter_grid(
    X_full, oof_true_full, top_feats_full,
    os.path.join(CONFIG['plots_dir'], "03_seattle_full_top_features_scatter.png"),
    max_plots=CONFIG['top_scatter_feats']
)

# PHYSICAL
X_phys, _, phys_feats = select_numeric_physical(dfS)
print(f"Seattle (PHYSICAL): {len(phys_feats)} features used")
met_phys, best_iters_phys, oof_true_phys, oof_pred_phys = run_cv_oof(X_phys, y, "Seattle (PHYSICAL)")
print_cv_results("Seattle (PHYSICAL)", met_phys, os.path.join(CONFIG['plots_dir'], "seattle_physical_cv_results.csv"))

bst_phys, imp_phys = train_full_and_importance(X_phys, y, best_rounds_full)
display_feature_importance(imp_phys, 10)
plot_feature_importance(imp_phys, "Seattle (PHYSICAL) — Top 10 Feature Importance",
                        top_n=10,
                        out_path=os.path.join(CONFIG['plots_dir'], "05_seattle_physical_feature_importance.png"))

plot_pred_vs_actual(oof_true_phys, oof_pred_phys,
                    "Seattle (PHYSICAL) — Out-of-Fold Predictions vs Actual",
                    os.path.join(CONFIG['plots_dir'], "02_seattle_physical_pred_vs_actual.png"))

# ONE scatter grid only
top_feats_phys = imp_phys['feature'].head(CONFIG['top_scatter_feats']).tolist()
plot_feature_scatter_grid(
    X_phys, oof_true_phys, top_feats_phys,
    os.path.join(CONFIG['plots_dir'], "08_seattle_physical_top_features_scatter.png"),
    max_plots=CONFIG['top_scatter_feats']
)

# --- Outputs ---
# 01_seattle_full_pred_vs_actual.png
# 02_seattle_physical_pred_vs_actual.png
# 03_seattle_full_top_features_scatter.png
# 04_seattle_full_feature_importance.png
# 05_seattle_physical_feature_importance.png
# 08_seattle_physical_top_features_scatter.png
# seattle_full_cv_results.csv
# seattle_physical_cv_results.csv


In [ ]:
# ===== King County — 5-Fold CV, Scatter Plots (grid only), Feature Counts + MAPE (King ONLY) =====
# Assumes df_king_enc exists and TARGET 'sale_price_usd' is log1p IN PLACE.

import os, numpy as np, pandas as pd, xgboost as xgb, matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from matplotlib.ticker import FuncFormatter

# -------------------- CONFIG --------------------
CONFIG = {
    'cv_folds': 5,
    'params': {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse',
        'eta': 0.093, 'max_depth': 7, 'min_child_weight': 5,
        'subsample': 0.609, 'colsample_bytree': 0.983,
        'reg_alpha': 0.145, 'reg_lambda': 1.283, 'gamma': 0.033,
        'tree_method': 'hist', 'grow_policy': 'lossguide', 'seed': 42,
    },
    'num_boost_round': 4453,
    'early_stopping_rounds': 150,
    'plots_dir': 'plots',
    'dpi': 220,
    'topn_fi': 30,
    'top_scatter_feats': 3   # match Seattle: THREE top features
}
os.makedirs(CONFIG['plots_dir'], exist_ok=True)
plt.rcParams['font.size'] = 11

TARGET = "sale_price_usd"  # already log1p upstream

PHYSICAL_FEATURES = [
    'year_built','year_renovated','lot_sqft',
    'total_living_sqft','main_floor_sqft','finished_basement_sqft','total_garage_sqft',
    'building_grade','basement_grade','home_condition',
    'number_of_stories','bedrooms',
    'full_bathrooms','three_quarter_bathrooms','half_bathrooms'
]

# -------------------- HELPERS (no pretty labels) --------------------
def money_short(x):
    x = float(x)
    return f"${x/1e6:.1f}M" if x>=1e6 else (f"${x/1e3:,.0f}k" if x>=1e3 else f"${x:,.0f}")

def _money_axis():
    return FuncFormatter(lambda t,_: f'${t/1e6:.1f}M' if t>=1e6 else f'${t/1e3:,.0f}k')

def select_numeric_full(df: pd.DataFrame):
    num_df = df.select_dtypes(include=['number','bool']).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors='coerce')
    def is_price_leak(col: str):
        lc = col.lower()
        return col != TARGET and (lc.startswith('sale_price') or lc.endswith('_price') or lc.endswith('price_usd') or lc=='log_price')
    feats = [c for c in num_df.columns if c != TARGET and not is_price_leak(c)]
    return num_df[feats].astype('float32'), num_df[TARGET].astype(float).values, feats

def select_numeric_physical(df: pd.DataFrame):
    num_df = df.select_dtypes(include=['number','bool']).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors='coerce')
    feats = [c for c in PHYSICAL_FEATURES if c in num_df.columns]
    return num_df[feats].astype('float32'), num_df[TARGET].astype(float).values, feats

def rmse_from_mse(mse):
    return float(np.sqrt(mse))

def run_cv_oof(X: pd.DataFrame, y: np.ndarray, label: str):
    kf = KFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=42)
    metrics, best_iters = [], []
    feat_names = list(map(str, X.columns))
    oof_pred = np.full_like(y, np.nan, dtype=float)
    for fold, (tr, va) in enumerate(kf.split(X), 1):
        print(f"\n{label} — Fold {fold}/{CONFIG['cv_folds']}")
        print("Training...")
        dtr = xgb.DMatrix(X.iloc[tr].values, label=y[tr], feature_names=feat_names)
        dva = xgb.DMatrix(X.iloc[va].values, label=y[va], feature_names=feat_names)
        evals_result = {}
        bst = xgb.train(
            params=CONFIG['params'],
            dtrain=dtr,
            num_boost_round=CONFIG['num_boost_round'],
            evals=[(dtr,'train'), (dva,'val')],
            early_stopping_rounds=CONFIG['early_stopping_rounds'],
            evals_result=evals_result,
            verbose_eval=False
        )
        v = np.array(evals_result['val']['rmse'])
        best_iter = int(np.argmin(v)) + 1
        best_iters.append(best_iter)
        try:
            y_pred_log = bst.predict(dva, iteration_range=(0, best_iter))
        except TypeError:
            y_pred_log = bst.predict(dva, ntree_limit=best_iter)

        rmse_log = rmse_from_mse(mean_squared_error(y[va], y_pred_log))
        mae_log  = float(mean_absolute_error(y[va], y_pred_log))
        r2_log   = float(r2_score(y[va], y_pred_log))
        y_true, y_pred = np.expm1(y[va]), np.expm1(y_pred_log)
        rmse_usd = rmse_from_mse(mean_squared_error(y_true, y_pred))
        mae_usd  = float(mean_absolute_error(y_true, y_pred))
        mape     = float(np.mean(np.abs((y_true - y_pred)/np.clip(y_true,1,None)))*100)

        print(f"  Fold {fold}: RMSE(log)={rmse_log:.3f}  MAE(log)={mae_log:.3f}  "
              f"R²={r2_log:.2f}  RMSE($)={money_short(rmse_usd)}  "
              f"MAE($)={money_short(mae_usd)}  MAPE={mape:.1f}%")

        metrics.append(dict(rmse_log=rmse_log, mae_log=mae_log, r2_log=r2_log,
                            rmse_usd=rmse_usd, mae_usd=mae_usd, mape=mape))
        oof_pred[va] = y_pred_log

    met_df = pd.DataFrame(metrics)
    return met_df, best_iters, y.astype(float), oof_pred

def print_cv_results(name: str, met: pd.DataFrame, out_csv_path: str):
    print(f"\n{name} — {CONFIG['cv_folds']}-FOLD RESULTS")
    for i, row in met.iterrows():
        print(f"  Fold {i+1}: RMSE(log)={row['rmse_log']:.3f}  MAE(log)={row['mae_log']:.3f}  "
              f"R²={row['r2_log']:.2f}  RMSE($)={money_short(row['rmse_usd'])}  "
              f"MAE($)={money_short(row['mae_usd'])}  MAPE={row['mape']:.1f}%")
    avg = met.mean(numeric_only=True)
    print(f"\n{name} — AVERAGE")
    print(f"  RMSE(log)={avg['rmse_log']:.3f}  MAE(log)={avg['mae_log']:.3f}  "
          f"R²={avg['r2_log']:.2f}  RMSE($)={money_short(avg['rmse_usd'])}  "
          f"MAE($)={money_short(avg['mae_usd'])}  MAPE={avg['mape']:.1f}%")
    met.assign(fold=np.arange(1, len(met)+1)).to_csv(out_csv_path, index=False)
    print(f"Saved CV table: {out_csv_path}")

def plot_pred_vs_actual(y_true_log, y_pred_log, title, out_path):
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    ymax = np.nanquantile(np.concatenate([y_true, y_pred]), 0.995)

    plt.figure(figsize=(7.0, 6.2), constrained_layout=True)
    plt.scatter(y_true, y_pred, s=8, alpha=0.35, color='steelblue')
    plt.plot([0, ymax], [0, ymax], 'k--', lw=1)
    plt.xlim(0, ymax); plt.ylim(0, ymax)
    ax = plt.gca()
    ax.xaxis.set_major_formatter(_money_axis())
    ax.yaxis.set_major_formatter(_money_axis())
    plt.xlabel("Actual Price (USD)", fontsize=12)
    plt.ylabel("Predicted Price (USD)", fontsize=12)
    # Simple, descriptive title (no metrics/tagline)
    plt.title(title, fontsize=13)
    plt.grid(True, alpha=0.3)
    plt.savefig(out_path, dpi=CONFIG['dpi'], bbox_inches='tight')
    plt.show()
    plt.close()

def train_full_and_importance(X: pd.DataFrame, y: np.ndarray, rounds: int):
    feat_names = list(map(str, X.columns))
    bst = xgb.train(params=CONFIG['params'],
                    dtrain=xgb.DMatrix(X.values, label=y, feature_names=feat_names),
                    num_boost_round=rounds, verbose_eval=False)
    g = pd.Series(bst.get_score(importance_type='gain'), dtype=float)
    imp = g.reindex(feat_names).fillna(0.0).reset_index()
    imp.columns = ['feature','gain']
    return bst, imp.sort_values('gain', ascending=False)

def plot_feature_importance(imp_df, title, top_n=10, out_path=None):
    """Feature importance as horizontal bar chart (no pretty labels)."""
    top_features = imp_df.head(top_n).copy()
    top_features = top_features.sort_values('gain', ascending=True)

    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(top_features)))
    plt.barh(range(len(top_features)), top_features['gain'], color=colors, alpha=0.7)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance (Gain)', fontsize=12)
    plt.title(title, fontsize=14, pad=20)
    plt.grid(True, alpha=0.3, axis='x')
    for i, v in enumerate(top_features['gain']):
        plt.text(v + max(top_features['gain'])*0.01, i, f'{v:.3f}', va='center', fontsize=10)
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=CONFIG['dpi'], bbox_inches='tight')
    plt.show()
    plt.close()

def plot_feature_scatter_grid(X, y_true_log, features, out_path, max_plots=3):
    """Grid: top features vs ACTUAL price (USD) — ONE image only (no individual plots)."""
    y_true = np.expm1(y_true_log)
    feats = [f for f in features if f in X.columns][:max_plots]
    if not feats:
        return

    cols = 3 if len(feats) == 3 else 2
    rows = int(np.ceil(len(feats)/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4.8, rows*4.2), constrained_layout=True)
    axes = np.ravel(axes) if hasattr(axes, "__len__") else [axes]

    for ax, f in zip(axes, feats):
        x_arr = pd.to_numeric(X[f], errors='coerce').to_numpy(dtype=float)
        ok = np.isfinite(x_arr) & np.isfinite(y_true)
        ax.scatter(x_arr[ok], y_true[ok], s=8, alpha=0.30, color='steelblue')
        ax.set_title(f, fontsize=11)           # raw feature name
        ax.set_xlabel(f)
        ax.yaxis.set_major_formatter(_money_axis())
        ax.set_ylabel("Price (USD)")
        ax.grid(True, alpha=0.3)

    for j in range(len(feats), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Top Features vs Actual Price (USD)", y=1.02, fontsize=12)
    plt.savefig(out_path, dpi=CONFIG['dpi'], bbox_inches='tight')
    plt.show()
    plt.close()

def display_feature_importance(imp_df, top_n=10):
    print(f"\nTop {top_n} Features by Gain:")
    print("-" * 60)
    for i, row in imp_df.head(top_n).iterrows():
        print(f"{i+1:2d}. {row['feature']:30s} {row['gain']:10.4f}")
    print("-" * 60)

# -------------------- RUN (King only; mirror Seattle outputs) --------------------
assert 'df_king_enc' in globals(), "df_king_enc not found."
assert TARGET in df_king_enc.columns, f"Missing {TARGET} in df_king_enc."
dfK = df_king_enc.copy()

# FULL — CV + OOF
X_full, y, full_feats = select_numeric_full(dfK)
print(f"King (FULL): {len(full_feats)} features used")
met_full, best_iters_full, oof_true_full, oof_pred_full = run_cv_oof(X_full, y, "King (FULL)")
print_cv_results("King (FULL)", met_full, os.path.join(CONFIG['plots_dir'], "king_full_cv_results.csv"))
median_rounds = int(np.median(best_iters_full)) if len(best_iters_full) else CONFIG['num_boost_round']
best_rounds_full = max(100, int(round(0.5*CONFIG['num_boost_round'] + 0.5*median_rounds)))
bst_full, imp_full = train_full_and_importance(X_full, y, best_rounds_full)

display_feature_importance(imp_full, 20)
plot_feature_importance(imp_full, "King (FULL) — Top 20 Feature Importance",
                        top_n=20,
                        out_path=os.path.join(CONFIG['plots_dir'], "04_king_full_feature_importance.png"))

plot_pred_vs_actual(
    oof_true_full, oof_pred_full,
    "King County (FULL model) — Scatter Plot (Predicted vs Actual)",
    os.path.join(CONFIG['plots_dir'], "01_king_full_pred_vs_actual.png")
)

# ONE image only: scatter grid of top 3 features (FULL) vs ACTUAL
top_feats_full = imp_full['feature'].head(CONFIG['top_scatter_feats']).tolist()
plot_feature_scatter_grid(
    X_full, oof_true_full, top_feats_full,
    os.path.join(CONFIG['plots_dir'], "03_king_full_top_features_scatter.png"),
    max_plots=CONFIG['top_scatter_feats']
)

# PHYSICAL — CV + OOF
X_phys, _, phys_feats = select_numeric_physical(dfK)
print(f"King (PHYSICAL): {len(phys_feats)} features used")
met_phys, best_iters_phys, oof_true_phys, oof_pred_phys = run_cv_oof(X_phys, y, "King (PHYSICAL)")
print_cv_results("King (PHYSICAL)", met_phys, os.path.join(CONFIG['plots_dir'], "king_physical_cv_results.csv"))

bst_phys, imp_phys = train_full_and_importance(X_phys, y, best_rounds_full)
display_feature_importance(imp_phys, 10)
plot_feature_importance(imp_phys, "King (PHYSICAL) — Top 10 Feature Importance",
                        top_n=10,
                        out_path=os.path.join(CONFIG['plots_dir'], "05_king_physical_feature_importance.png"))

plot_pred_vs_actual(
    oof_true_phys, oof_pred_phys,
    "King County (PHYSICAL model) — Scatter Plot (Predicted vs Actual)",
    os.path.join(CONFIG['plots_dir'], "02_king_physical_pred_vs_actual.png")
)

# ONE image only: scatter grid of top 3 features (PHYSICAL) vs ACTUAL
top_feats_phys = imp_phys['feature'].head(CONFIG['top_scatter_feats']).tolist()
plot_feature_scatter_grid(
    X_phys, oof_true_phys, top_feats_phys,
    os.path.join(CONFIG['plots_dir'], "08_king_physical_top_features_scatter.png"),
    max_plots=CONFIG['top_scatter_feats']
)

# Files produced (mirrors Seattle; no per-feature scatter images, no pretty labels):
# 01_king_full_pred_vs_actual.png
# 02_king_physical_pred_vs_actual.png
# 03_king_full_top_features_scatter.png
# 04_king_full_feature_importance.png
# 05_king_physical_feature_importance.png
# 08_king_physical_top_features_scatter.png
# king_full_cv_results.csv
# king_physical_cv_results.csv


In [ ]:
# TabNet dependencies are installed in the local Anaconda environment.
print("pytorch-tabnet and pytorch-lightning dependency cell skipped locally.")


In [ ]:
# === Seattle — TabNet (FULL: normal + residuals; PHYSICAL: metrics + FI) ===
# Requires: df_seattle_enc exists and 'sale_price_usd' is log1p IN PLACE.

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from matplotlib.ticker import FuncFormatter
from pytorch_tabnet.tab_model import TabNetRegressor

# -------------------- CONFIG --------------------
TARGET = "sale_price_usd"
N_SPLITS = 2  # local execution setting; use 5 for final research run

# Large budget + early stopping (no epoch logs shown)
MAX_EPOCHS = 10  # local execution setting; use 200 for final research run
PATIENCE   = 3   # local execution setting; use 20 for final research run

BATCH_SIZE = 16384
VIRTUAL_BATCH_SIZE = 2048
LR = 2e-2
WEIGHT_DECAY = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PLOTS_DIR = "plots"
DPI = 220
TOPN_FI = 20            # match King/Seattle style (Top 20 plot)
RESIDUAL_TOP_FEATS = 3  # single grid, only three features
os.makedirs(PLOTS_DIR, exist_ok=True)
plt.rcParams["font.size"] = 11

# -------------------- PHYSICAL feature list (same as XGB) --------------------
PHYSICAL_FEATURES = [
    "year_built","year_renovated","lot_sqft",
    "total_living_sqft","main_floor_sqft","finished_basement_sqft","total_garage_sqft",
    "building_grade","basement_grade","home_condition",
    "number_of_stories","bedrooms",
    "full_bathrooms","three_quarter_bathrooms","half_bathrooms"
]

# -------------------- Helpers --------------------
def select_numeric_full(df: pd.DataFrame):
    num_df = df.select_dtypes(include=["number","bool"]).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
    def is_leak(col: str):
        lc = col.lower()
        return col != TARGET and (lc.startswith("sale_price") or lc.endswith("_price") or lc.endswith("price_usd") or lc=="log_price")
    feats = [c for c in num_df.columns if c != TARGET and not is_leak(c)]
    return num_df[feats].astype("float32"), num_df[TARGET].astype("float32").values, feats

def select_numeric_physical(df: pd.DataFrame):
    num_df = df.select_dtypes(include=["number","bool"]).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
    feats = [c for c in PHYSICAL_FEATURES if c in num_df.columns]
    return num_df[feats].astype("float32"), num_df[TARGET].astype("float32").values, feats

def metrics_log_and_usd(y_true_log, y_pred_log):
    rmse_log = float(np.sqrt(mean_squared_error(y_true_log, y_pred_log)))
    mae_log  = float(mean_absolute_error(y_true_log, y_pred_log))
    r2_log   = float(r2_score(y_true_log, y_pred_log))
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    rmse_usd = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae_usd  = float(mean_absolute_error(y_true, y_pred))
    mape     = float(np.mean(np.abs((y_true - y_pred)/np.clip(y_true, 1, None)))*100)
    return dict(rmse_log=rmse_log, mae_log=mae_log, r2_log=r2_log,
                rmse_usd=rmse_usd, mae_usd=mae_usd, mape=mape)

def money_short(x):
    x = float(x)
    return f"${x/1e6:.2f}M" if x>=1e6 else (f"${x/1e3:,.0f}k" if x>=1e3 else f"${x:,.0f}")

def _money_axis():
    return FuncFormatter(lambda t,_: f'${t/1e6:.1f}M' if t>=1e6 else f'${t/1e3:,.0f}k')

# -------------------- CV (no curves) + OOF + FI --------------------
def run_tabnet_cv_oof(Xdf: pd.DataFrame, y_log: np.ndarray, label: str):
    feat_names = list(map(str, Xdf.columns))
    print(f"\n{label}: {Xdf.shape[1]} features | folds={N_SPLITS}")
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

    fold_metrics = []
    imp_sum = pd.Series(0.0, index=feat_names, dtype=float)
    oof_pred = np.full_like(y_log, np.nan, dtype=float)
    oof_true = y_log.astype(float).copy()

    for fold, (tr, va) in enumerate(kf.split(Xdf), 1):
        Xtr_df, Xva_df = Xdf.iloc[tr], Xdf.iloc[va]
        ytr, yva = y_log[tr], y_log[va]

        scaler = RobustScaler()
        Xtr = scaler.fit_transform(Xtr_df.values)
        Xva = scaler.transform(Xva_df.values)

        model = TabNetRegressor(
            n_d=8, n_a=8, n_steps=3, gamma=1.3,
            n_independent=2, n_shared=2,
            lambda_sparse=1e-4,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=LR, weight_decay=WEIGHT_DECAY),
            mask_type="entmax",
            clip_value=2.0,
            seed=42,
            device_name=DEVICE,
            verbose=0   # suppress epoch logs
        )

        model.fit(
            Xtr, ytr.reshape(-1,1),
            eval_set=[(Xva, yva.reshape(-1,1))],
            eval_name=["valid"],
            eval_metric=["rmse"],
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            batch_size=BATCH_SIZE,
            virtual_batch_size=VIRTUAL_BATCH_SIZE,
            num_workers=0,
            drop_last=False
        )

        y_pred_log = model.predict(Xva).reshape(-1)
        m = metrics_log_and_usd(yva, y_pred_log)
        print(f"{label} | Fold {fold}: RMSE(log)={m['rmse_log']:.4f}  MAE(log)={m['mae_log']:.4f}  "
              f"R²={m['r2_log']:.3f}  |  RMSE($)={money_short(m['rmse_usd'])}  "
              f"MAE($)={money_short(m['mae_usd'])}  MAPE={m['mape']:.1f}%")
        fold_metrics.append(m)
        oof_pred[va] = y_pred_log

        fi = pd.Series(model.feature_importances_, index=feat_names, dtype=float)
        imp_sum = imp_sum.add(fi, fill_value=0.0)

    met = pd.DataFrame(fold_metrics)

    # Best fold + averages (printed like XGB version)
    best_idx = int(met["rmse_usd"].idxmin())
    mb = met.loc[best_idx].to_dict()
    print(f"\n{label} — BEST FOLD (by RMSE $)  |  fold #{best_idx+1}")
    print(f"  log: RMSE={mb['rmse_log']:.4f} | MAE={mb['mae_log']:.4f} | R²={mb['r2_log']:.3f}")
    print(f"  $  : RMSE={money_short(mb['rmse_usd'])} | MAE={money_short(mb['mae_usd'])} | MAPE={mb['mape']:.1f}%")

    print(f"\n=== {label} | {N_SPLITS}-Fold Averages ===")
    print(f"RMSE(log): {met['rmse_log'].mean():.4f}   MAE(log): {met['mae_log'].mean():.4f}   R²: {met['r2_log'].mean():.3f}")
    print(f"RMSE($):   {money_short(met['rmse_usd'].mean())}   MAE($): {money_short(met['mae_usd'].mean())}   "
          f"MAPE: {met['mape'].mean():.1f}%")

    imp_df = imp_sum.sort_values(ascending=False).reset_index()
    imp_df.columns = ["feature","importance_sum"]
    return met, imp_df, oof_true, oof_pred

# -------------------- Plotters --------------------
def plot_pred_vs_actual(y_true_log, y_pred_log, title, out_path):
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    ymax = np.nanquantile(np.concatenate([y_true, y_pred]), 0.995)

    plt.figure(figsize=(6.8, 6.1), constrained_layout=True)
    plt.scatter(y_true, y_pred, s=8, alpha=0.35, color="steelblue")
    plt.plot([0, ymax], [0, ymax], "k--", lw=1)
    plt.xlim(0, ymax); plt.ylim(0, ymax)
    ax = plt.gca(); ax.xaxis.set_major_formatter(_money_axis()); ax.yaxis.set_major_formatter(_money_axis())
    plt.xlabel("Actual Price (USD)"); plt.ylabel("Predicted Price (USD)")
    plt.title(f"{title}\nRMSE={money_short(rmse)} | MAE={money_short(mae)} | R²={r2:.2f}")
    plt.grid(True, alpha=0.3)
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {out_path}")

def plot_residuals_vs_actual(y_true_log, y_pred_log, out_path, bins=30):
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    resid = y_true - y_pred
    xmax = np.nanquantile(y_true, 0.995)
    ymax = np.nanquantile(np.abs(resid), 0.995)

    plt.figure(figsize=(7.2, 5.2), constrained_layout=True)
    plt.scatter(y_true, resid, s=6, alpha=0.25)
    qs = np.linspace(0, 1, bins+1); edges = np.unique(np.quantile(y_true, qs))
    if len(edges) >= 3:
        idx = np.clip(np.digitize(y_true, edges)-1, 0, len(edges)-2)
        centers = 0.5*(edges[:-1] + edges[1:])
        means = pd.Series(resid).groupby(idx).mean().values
        plt.plot(centers[:len(means)], means, lw=2, label="binned mean residual")
    plt.axhline(0, color="k", lw=1, linestyle="--")
    ax = plt.gca(); ax.xaxis.set_major_formatter(_money_axis()); ax.yaxis.set_major_formatter(_money_axis())
    plt.xlim(0, xmax); plt.ylim(-ymax, ymax)
    plt.title("Residuals vs Actual Price (USD)\nResidual = actual − predicted")
    plt.xlabel("Actual Price (USD)"); plt.ylabel("Residual (USD)")
    plt.legend(loc="upper right", fontsize=9)
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {out_path}")

def save_fi_plot_csv(imp_df: pd.DataFrame, title: str, png_path: str, csv_path: str, topn=20):
    top = imp_df.head(topn).copy()
    top.to_csv(csv_path, index=False)
    # Seattle-style: viridis + value labels
    top_sorted = top.sort_values("importance_sum", ascending=True)
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(top_sorted)))
    plt.barh(range(len(top_sorted)), top_sorted["importance_sum"], color=colors, alpha=0.7)
    plt.yticks(range(len(top_sorted)), top_sorted["feature"])
    plt.xlabel("Feature Importance (sum over folds)"); plt.title(title, pad=20)
    plt.grid(True, alpha=0.3, axis="x")
    maxv = top_sorted["importance_sum"].max() if len(top_sorted) else 1.0
    for i, v in enumerate(top_sorted["importance_sum"]):
        plt.text(v + maxv*0.01, i, f"{v:.3f}", va="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(png_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {png_path} | {csv_path}")

def plot_residuals_vs_top_features(X, y_true_log, y_pred_log, features, out_path, bins=20, max_plots=3):
    """Single GRID image of residuals vs top features (limited to 3)."""
    resid = np.expm1(y_true_log) - np.expm1(y_pred_log)
    feats = [f for f in features if f in X.columns][:max_plots]
    if not feats: return
    cols = 3 if len(feats) == 3 else 2
    rows = int(np.ceil(len(feats)/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4.8, rows*3.8), constrained_layout=True)
    axes = np.ravel(axes) if hasattr(axes, "__len__") else [axes]
    for ax, f in zip(axes, feats):
        x = pd.to_numeric(X[f], errors="coerce")
        ok = x.notna() & np.isfinite(x) & np.isfinite(resid)
        x, r = x[ok], resid[ok]
        if x.nunique(dropna=True) <= 10:
            order = sorted(x.dropna().unique())
            data = [r[x==v] for v in order]
            ax.boxplot(data, showfliers=False)
            ax.set_xticks(range(1, len(order)+1)); ax.set_xticklabels(order)
            ax.axhline(0, color="k", lw=1, linestyle="--")
            ax.set_ylabel("Residual (USD)")
        else:
            qs = np.linspace(0, 1, bins+1); edges = np.unique(np.quantile(x, qs))
            if len(edges) >= 3:
                idx = np.clip(np.digitize(x, edges)-1, 0, len(edges)-2)
                centers = 0.5*(edges[:-1] + edges[1:])
                means = pd.Series(r).groupby(idx).mean().values
                ax.plot(centers[:len(means)], means, lw=2)
                ax.axhline(0, color="k", lw=1, linestyle="--")
                ax.set_ylabel("Residual (USD)")
        ax.set_title(f.replace('_',' ').title(), fontsize=11)
        ax.set_xlabel(f)
    for j in range(len(feats), len(axes)): axes[j].set_visible(False)
    fig.suptitle("Seattle (TabNet FULL) — Residuals vs Top Features\nResidual = actual − predicted",
                 y=1.02, fontsize=12)
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {out_path}")

# -------------------- RUN (Seattle) --------------------
assert "df_seattle_enc" in globals(), "df_seattle_enc missing."
assert TARGET in df_seattle_enc.columns, f"Missing '{TARGET}'. Ensure log1p in-place."
dfS = df_seattle_enc.copy()

# FULL
X_full, y_full, full_feats = select_numeric_full(dfS)
full_met, full_imp, sea_oof_true, sea_oof_pred = run_tabnet_cv_oof(X_full, y_full, label="Seattle (FULL, TabNet)")

# FULL outputs: OOF scatter, residuals, FI (PNG+CSV), OOF CSV
plot_pred_vs_actual(sea_oof_true, sea_oof_pred,
                    "Seattle (TabNet FULL) — Out-of-Fold Predictions vs Actual",
                    os.path.join(PLOTS_DIR, "sea_tabnet_full_pred_vs_actual.png"))

plot_residuals_vs_actual(sea_oof_true, sea_oof_pred,
                         out_path=os.path.join(PLOTS_DIR, "sea_tabnet_full_residuals_vs_actual.png"),
                         bins=30)

top_feats_full = full_imp["feature"].head(RESIDUAL_TOP_FEATS).tolist()
plot_residuals_vs_top_features(
    X_full, sea_oof_true, sea_oof_pred, top_feats_full,
    out_path=os.path.join(PLOTS_DIR, "sea_tabnet_full_residuals_vs_top_features.png"),
    bins=20, max_plots=RESIDUAL_TOP_FEATS
)

save_fi_plot_csv(
    full_imp, "Seattle — TabNet FULL Top 20 Feature Importance",
    os.path.join(PLOTS_DIR, "sea_tabnet_full_top_importance.png"),
    os.path.join(PLOTS_DIR, "sea_tabnet_full_top_importance.csv"),
    topn=TOPN_FI
)

oof_df = pd.DataFrame({
    "actual_usd": np.expm1(sea_oof_true),
    "pred_usd":   np.expm1(sea_oof_pred)
})
oof_df["residual_usd"] = oof_df["actual_usd"] - oof_df["pred_usd"]
oof_df["abs_pct_err"]  = np.where(oof_df["actual_usd"]>0,
                                  np.abs(oof_df["residual_usd"]/oof_df["actual_usd"])*100, np.nan)
oof_df.to_csv(os.path.join(PLOTS_DIR, "sea_tabnet_full_oof_predictions.csv"), index=False)

# PHYSICAL (metrics + FI PNG+CSV; no other images)
X_phys, y_phys, phys_feats = select_numeric_physical(dfS)
phys_met, phys_imp, _, _ = run_tabnet_cv_oof(X_phys, y_phys, label="Seattle (PHYSICAL, TabNet)")

save_fi_plot_csv(
    phys_imp, "Seattle — TabNet PHYSICAL Top 20 Feature Importance",
    os.path.join(PLOTS_DIR, "sea_tabnet_physical_top_importance.png"),
    os.path.join(PLOTS_DIR, "sea_tabnet_physical_top_importance.csv"),
    topn=TOPN_FI
)

# ---- Files produced ----
# sea_tabnet_full_pred_vs_actual.png
# sea_tabnet_full_residuals_vs_actual.png
# sea_tabnet_full_residuals_vs_top_features.png   (single grid, 3 features)
# sea_tabnet_full_top_importance.png / sea_tabnet_full_top_importance.csv  (Top 20)
# sea_tabnet_full_oof_predictions.csv
# sea_tabnet_physical_top_importance.png / sea_tabnet_physical_top_importance.csv  (Top 20)




In [ ]:
# === King County — TabNet (FULL: OOF scatter w/ simple title; residuals; FI) ===
# df_king_enc exists and 'sale_price_usd' is log1p IN PLACE upstream.

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from matplotlib.ticker import FuncFormatter
from pytorch_tabnet.tab_model import TabNetRegressor

# -------------------- CONFIG --------------------
TARGET = "sale_price_usd"
N_SPLITS = 2  # local execution setting; use 5 for final research run

MAX_EPOCHS = 10  # local execution setting; use 200 for final research run
PATIENCE   = 3   # local execution setting; use 20 for final research run

BATCH_SIZE = 16384
VIRTUAL_BATCH_SIZE = 2048
LR = 2e-2
WEIGHT_DECAY = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PLOTS_DIR = "plots"
DPI = 220
TOPN_FI = 20            # Top 20 FI (plot + CSV)
RESIDUAL_TOP_FEATS = 3  # one residuals grid of 3 features
os.makedirs(PLOTS_DIR, exist_ok=True)
plt.rcParams["font.size"] = 11

# -------------------- PHYSICAL features (match XGB) --------------------
PHYSICAL_FEATURES = [
    "year_built","year_renovated","lot_sqft",
    "total_living_sqft","main_floor_sqft","finished_basement_sqft","total_garage_sqft",
    "building_grade","basement_grade","home_condition",
    "number_of_stories","bedrooms",
    "full_bathrooms","three_quarter_bathrooms","half_bathrooms"
]

# -------------------- Helpers --------------------
def select_numeric_full(df: pd.DataFrame):
    """All numeric except target and price-like leakers."""
    num_df = df.select_dtypes(include=["number","bool"]).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")

    def is_leak(col: str):
        lc = col.lower()
        return col != TARGET and (lc.startswith("sale_price") or lc.endswith("_price") or lc.endswith("price_usd") or lc=="log_price")

    feats = [c for c in num_df.columns if c != TARGET and not is_leak(c)]
    X = num_df[feats].astype("float32")
    y = num_df[TARGET].astype("float32").values
    return X, y, feats

def select_numeric_physical(df: pd.DataFrame):
    num_df = df.select_dtypes(include=["number","bool"]).copy()
    for c in num_df.columns:
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
    feats = [c for c in PHYSICAL_FEATURES if c in num_df.columns]
    X = num_df[feats].astype("float32")
    y = num_df[TARGET].astype("float32").values
    return X, y, feats

def metrics_log_and_usd(y_true_log, y_pred_log):
    rmse_log = float(np.sqrt(mean_squared_error(y_true_log, y_pred_log)))
    mae_log  = float(mean_absolute_error(y_true_log, y_pred_log))
    r2_log   = float(r2_score(y_true_log, y_pred_log))
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    rmse_usd = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae_usd  = float(mean_absolute_error(y_true, y_pred))
    mape     = float(np.mean(np.abs((y_true - y_pred)/np.clip(y_true, 1, None)))*100)
    return dict(rmse_log=rmse_log, mae_log=mae_log, r2_log=r2_log,
                rmse_usd=rmse_usd, mae_usd=mae_usd, mape=mape)

def money_short(x):
    x = float(x)
    return f"${x/1e6:.2f}M" if x>=1e6 else (f"${x/1e3:,.0f}k" if x>=1e3 else f"${x:,.0f}")

def _money_axis():
    return FuncFormatter(lambda t,_: f'${t/1e6:.1f}M' if t>=1e6 else f'${t/1e3:,.0f}k')

# -------------------- CV (no curves) + OOF + FI --------------------
def run_tabnet_cv_oof(Xdf: pd.DataFrame, y_log: np.ndarray, label: str):
    """5-fold TabNet CV with per-fold prints + best + averages. Returns metrics, FI, OOF arrays."""
    feat_names = list(map(str, Xdf.columns))
    print(f"\n{label}: {Xdf.shape[1]} features | folds={N_SPLITS}")
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

    fold_metrics = []
    imp_sum = pd.Series(0.0, index=feat_names, dtype=float)
    oof_pred = np.full_like(y_log, np.nan, dtype=float)
    oof_true = y_log.astype(float).copy()

    for fold, (tr, va) in enumerate(kf.split(Xdf), 1):
        Xtr_df, Xva_df = Xdf.iloc[tr], Xdf.iloc[va]
        ytr, yva = y_log[tr], y_log[va]

        scaler = RobustScaler()
        Xtr, Xva = scaler.fit_transform(Xtr_df.values), scaler.transform(Xva_df.values)

        model = TabNetRegressor(
            n_d=8, n_a=8, n_steps=3, gamma=1.3,
            n_independent=2, n_shared=2,
            lambda_sparse=1e-4,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=LR, weight_decay=WEIGHT_DECAY),
            mask_type="entmax", clip_value=2.0, seed=42, device_name=DEVICE,
            verbose=0   # suppress epoch logs
        )

        model.fit(
            Xtr, ytr.reshape(-1,1),
            eval_set=[(Xva, yva.reshape(-1,1))],
            eval_name=["valid"], eval_metric=["rmse"],
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            batch_size=BATCH_SIZE, virtual_batch_size=VIRTUAL_BATCH_SIZE,
            num_workers=0, drop_last=False
        )

        y_pred_log = model.predict(Xva).reshape(-1)
        m = metrics_log_and_usd(yva, y_pred_log)
        print(f"{label} | Fold {fold}: RMSE(log)={m['rmse_log']:.4f}  MAE(log)={m['mae_log']:.4f}  "
              f"R²={m['r2_log']:.3f}  |  RMSE($)={money_short(m['rmse_usd'])}  "
              f"MAE($)={money_short(m['mae_usd'])}  MAPE={m['mape']:.1f}%")
        fold_metrics.append(m)
        oof_pred[va] = y_pred_log

        fi = pd.Series(model.feature_importances_, index=feat_names, dtype=float)
        imp_sum = imp_sum.add(fi, fill_value=0.0)

    met = pd.DataFrame(fold_metrics)

    # Best fold + averages (mirror Seattle)
    best_idx = int(met["rmse_usd"].idxmin())
    mb = met.loc[best_idx].to_dict()
    print(f"\n{label} — BEST FOLD (by RMSE $)  |  fold #{best_idx+1}")
    print(f"  log: RMSE={mb['rmse_log']:.4f} | MAE={mb['mae_log']:.4f} | R²={mb['r2_log']:.3f}")
    print(f"  $  : RMSE={money_short(mb['rmse_usd'])} | MAE={money_short(mb['mae_usd'])} | MAPE={mb['mape']:.1f}%")

    print(f"\n=== {label} | {N_SPLITS}-Fold Averages ===")
    print(f"RMSE(log): {met['rmse_log'].mean():.4f}   MAE(log): {met['mae_log'].mean():.4f}   R²: {met['r2_log'].mean():.3f}")
    print(f"RMSE($):   {money_short(met['rmse_usd'].mean())}   MAE($): {money_short(met['mae_usd'].mean())}   "
          f"MAPE: {met['mape'].mean():.1f}%")

    imp_df = imp_sum.sort_values(ascending=False).reset_index()
    imp_df.columns = ["feature","importance_sum"]
    return met, imp_df, oof_true, oof_pred

# -------------------- Plotters (simple-title scatter) --------------------
def plot_pred_vs_actual(y_true_log, y_pred_log, title, out_path):
    """Simple-title scatter: Predicted vs Actual (no metrics in title)."""
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    ymax = np.nanquantile(np.concatenate([y_true, y_pred]), 0.995)

    plt.figure(figsize=(6.8, 6.1), constrained_layout=True)
    plt.scatter(y_true, y_pred, s=8, alpha=0.35)
    plt.plot([0, ymax], [0, ymax], "k--", lw=1)
    plt.xlim(0, ymax); plt.ylim(0, ymax)
    ax = plt.gca(); ax.xaxis.set_major_formatter(_money_axis()); ax.yaxis.set_major_formatter(_money_axis())
    plt.xlabel("Actual Price (USD)"); plt.ylabel("Predicted Price (USD)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {out_path}")

def plot_residuals_vs_actual(y_true_log, y_pred_log, out_path, bins=30):
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    resid = y_true - y_pred
    xmax = np.nanquantile(y_true, 0.995)
    ymax = np.nanquantile(np.abs(resid), 0.995)

    plt.figure(figsize=(7.2, 5.2), constrained_layout=True)
    plt.scatter(y_true, resid, s=6, alpha=0.25)
    qs = np.linspace(0, 1, bins+1); edges = np.unique(np.quantile(y_true, qs))
    if len(edges) >= 3:
        idx = np.clip(np.digitize(y_true, edges)-1, 0, len(edges)-2)
        centers = 0.5*(edges[:-1] + edges[1:])
        means = pd.Series(resid).groupby(idx).mean().values
        plt.plot(centers[:len(means)], means, lw=2, label="binned mean residual")
    plt.axhline(0, color="k", lw=1, linestyle="--")
    ax = plt.gca(); ax.xaxis.set_major_formatter(_money_axis()); ax.yaxis.set_major_formatter(_money_axis())
    plt.xlim(0, xmax); plt.ylim(-ymax, ymax)
    plt.title("Residuals vs Actual Price (USD)\nResidual = actual − predicted")
    plt.xlabel("Actual Price (USD)"); plt.ylabel("Residual (USD)")
    plt.legend(loc="upper right", fontsize=9)
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {out_path}")

def save_fi_plot_csv(imp_df: pd.DataFrame, title: str, png_path: str, csv_path: str, topn=20):
    top = imp_df.head(topn).copy()
    top.to_csv(csv_path, index=False)
    # Seattle-style: viridis + value labels
    top_sorted = top.sort_values("importance_sum", ascending=True)
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(top_sorted)))
    plt.barh(range(len(top_sorted)), top_sorted["importance_sum"], color=colors, alpha=0.7)
    plt.yticks(range(len(top_sorted)), top_sorted["feature"])
    plt.xlabel("Feature Importance (sum over folds)"); plt.title(title, pad=20)
    plt.grid(True, alpha=0.3, axis="x")
    maxv = top_sorted["importance_sum"].max() if len(top_sorted) else 1.0
    for i, v in enumerate(top_sorted["importance_sum"]):
        plt.text(v + maxv*0.01, i, f"{v:.3f}", va="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(png_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {png_path} | {csv_path}")

def plot_residuals_vs_top_features(X, y_true_log, y_pred_log, features, out_path, bins=20, max_plots=3):
    """Single GRID image of residuals vs top features (limited to 3)."""
    resid = np.expm1(y_true_log) - np.expm1(y_pred_log)
    feats = [f for f in features if f in X.columns][:max_plots]
    if not feats: return
    cols = 3 if len(feats) == 3 else 2
    rows = int(np.ceil(len(feats)/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4.8, rows*3.8), constrained_layout=True)
    axes = np.ravel(axes) if hasattr(axes, "__len__") else [axes]
    for ax, f in zip(axes, feats):
        x = pd.to_numeric(X[f], errors="coerce")
        ok = x.notna() & np.isfinite(x) & np.isfinite(resid)
        x, r = x[ok], resid[ok]
        if x.nunique(dropna=True) <= 10:
            order = sorted(x.dropna().unique())
            data = [r[x==v] for v in order]
            ax.boxplot(data, showfliers=False)
            ax.set_xticks(range(1, len(order)+1)); ax.set_xticklabels(order)
            ax.axhline(0, color="k", lw=1, linestyle="--")
            ax.set_ylabel("Residual (USD)")
        else:
            qs = np.linspace(0, 1, bins+1); edges = np.unique(np.quantile(x, qs))
            if len(edges) >= 3:
                idx = np.clip(np.digitize(x, edges)-1, 0, len(edges)-2)
                centers = 0.5*(edges[:-1] + edges[1:])
                means = pd.Series(r).groupby(idx).mean().values
                ax.plot(centers[:len(means)], means, lw=2)
                ax.axhline(0, color="k", lw=1, linestyle="--")
                ax.set_ylabel("Residual (USD)")
        ax.set_title(f.replace('_',' ').title(), fontsize=11)
        ax.set_xlabel(f)
    for j in range(len(feats), len(axes)): axes[j].set_visible(False)
    fig.suptitle("King County (TabNet FULL) — Residuals vs Top Features\nResidual = actual − predicted",
                 y=1.02, fontsize=12)
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight"); plt.show()
    print(f"Saved: {out_path}")

# -------------------- RUN (King County) --------------------
assert "df_king_enc" in globals(), "df_king_enc missing."
assert TARGET in df_king_enc.columns, f"Missing '{TARGET}'. Ensure log1p in-place."

dfK = df_king_enc.copy()

# FULL (all numeric except target & leakers)
X_full, y_full, full_feats = select_numeric_full(dfK)
full_met, full_imp, kc_oof_true, kc_oof_pred = run_tabnet_cv_oof(X_full, y_full, label="King County (FULL, TabNet)")

# FULL outputs: OOF scatter (simple title/name), residuals, FI (PNG+CSV), OOF CSV
plot_pred_vs_actual(
    kc_oof_true, kc_oof_pred,
    title="TabNet King County — Scatter Plot (Predicted vs Actual)",
    out_path=os.path.join(PLOTS_DIR, "TabNet_KingCounty_scatter_plot.png")
)

plot_residuals_vs_actual(
    kc_oof_true, kc_oof_pred,
    out_path=os.path.join(PLOTS_DIR, "kc_tabnet_full_residuals_vs_actual.png"),
    bins=30
)

top_feats_full = full_imp["feature"].head(RESIDUAL_TOP_FEATS).tolist()
plot_residuals_vs_top_features(
    X_full, kc_oof_true, kc_oof_pred, top_feats_full,
    out_path=os.path.join(PLOTS_DIR, "kc_tabnet_full_residuals_vs_top_features.png"),
    bins=20, max_plots=RESIDUAL_TOP_FEATS
)

save_fi_plot_csv(
    full_imp, "King County — TabNet FULL Top 20 Feature Importance",
    os.path.join(PLOTS_DIR, "kc_tabnet_full_top_importance.png"),
    os.path.join(PLOTS_DIR, "kc_tabnet_full_top_importance.csv"),
    topn=TOPN_FI
)

# OOF table (Actual/Pred/Residual/APE) for FULL
oof_df = pd.DataFrame({
    "actual_usd": np.expm1(kc_oof_true),
    "pred_usd":   np.expm1(kc_oof_pred)
})
oof_df["residual_usd"] = oof_df["actual_usd"] - oof_df["pred_usd"]
oof_df["abs_pct_err"]  = np.where(oof_df["actual_usd"]>0,
                                  np.abs(oof_df["residual_usd"]/oof_df["actual_usd"])*100, np.nan)
oof_df.to_csv(os.path.join(PLOTS_DIR, "kc_tabnet_full_oof_predictions.csv"), index=False)

# PHYSICAL (metrics + FI PNG+CSV; no other images)
X_phys, y_phys, phys_feats = select_numeric_physical(dfK)
phys_met, phys_imp, _, _ = run_tabnet_cv_oof(X_phys, y_phys, label="King County (PHYSICAL, TabNet)")

save_fi_plot_csv(
    phys_imp, "King County — TabNet PHYSICAL Top 20 Feature Importance",
    os.path.join(PLOTS_DIR, "kc_tabnet_physical_top_importance.png"),
    os.path.join(PLOTS_DIR, "kc_tabnet_physical_top_importance.csv"),
    topn=TOPN_FI
)

# ---- Files produced (mirrors Seattle) ----
# TabNet_KingCounty_scatter_plot.png
# kc_tabnet_full_residuals_vs_actual.png
# kc_tabnet_full_residuals_vs_top_features.png   (single grid, 3 features)
# kc_tabnet_full_top_importance.png / kc_tabnet_full_top_importance.csv  (Top 20)
# kc_tabnet_full_oof_predictions.csv             (actual/pred/residual/APE)
# kc_tabnet_physical_top_importance.png / kc_tabnet_physical_top_importance.csv  (Top 20)




---

# At a Glance (for any reader)

**Best overall:** XGBoost (Full).  
**Seattle:** R² ≈ 0.92, MAPE ≈ 8.7% (structure + location).  
**King County:** R² ≈ 0.96, MAPE ≈ 7.3%.  
**Physical-only baseline:** explains ~60% of price; location features provide the big lift.

**How to read the metrics**
- **R²** — share of price variation explained (higher is better).  
- **RMSE(log)** & **MAE(log)** — average error in log space (lower is better).  
- **MAPE** — average percentage error (lower is better; easy to interpret).  

**What to look at in this notebook**
- Cross‑validated results tables (Seattle & King County; Full vs Physical).  
- Actual vs predicted scatter plots for the best models (tight 45° fit).  
- Top‑20 feature importance (XGBoost Full): build quality, interior size, age/renovation, lot & layout.

> TL;DR — Location features are essential. Physical features alone get ~60% of the way; adding location lifts accuracy into the 0.9+ R² range. XGBoost is the most accurate across regions.
